# 财务绩效预测 — 最终提交 Notebook

**课程:** 财务绩效预测  
**环境:** Python (QuantEnv)  
**说明:** 本 Notebook 完全自包含，仅依赖 `train.csv`、`test.csv`、`sample_submission.csv` 三个原始输入文件。所有特征工程、交叉验证、模型训练、融合、制图和 submission 逻辑均内嵌于此。

## 目录

1. 环境导入与常量
2. 数据读取与结构检查
3. EDA — 数据质量分析
4. EDA — 图表
5. 特征工程
6. GroupKFold 交叉验证设置
7. 基线模型 B0–B4
8. 机器学习对照模型 — Ridge
9. CatBoost 数据准备
10. CatBoost 直接预测 (M3a-M3d)
11. CatBoost 残差预测 (M4)
12. OOF 融合
13. 会计恒等式后处理
14. 模型结果比较
15. 生成 submission.csv
16. 导出同源结果文件
17. 最终格式校验

## 0. 环境导入与常量

In [ ]:
from __future__ import annotations

import hashlib
import itertools
import json
import math
import os
import re
import time
import warnings
from collections.abc import Iterator, Mapping, Sequence
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
from catboost import CatBoostRegressor
from sklearn.compose import ColumnTransformer
from sklearn.ensemble import HistGradientBoostingRegressor
from sklearn.impute import SimpleImputer
from sklearn.linear_model import Ridge
from sklearn.metrics import r2_score
from sklearn.model_selection import GroupKFold
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import OneHotEncoder, OrdinalEncoder, RobustScaler

warnings.filterwarnings("ignore", category=FutureWarning)
warnings.filterwarnings("ignore", category=UserWarning)

_NOTEBOOK_START_TIME = time.perf_counter()

### 0.1 项目常量与路径

In [ ]:
# ── Paths ──────────────────────────────────────────────────────────────
ROOT = Path.cwd().resolve()
RESULTS = ROOT / "results"
TABLES = RESULTS / "tables"
CV_SCORES = RESULTS / "cv_scores"
OOF_DIR = RESULTS / "oof"
PREDICTIONS_DIR = RESULTS / "predictions"
MODELS_DIR = RESULTS / "models"
FIGURES = ROOT / "figures"
DELIVERABLES = ROOT / "deliverables"
CONFIGS = ROOT / "configs"

for path in [RESULTS, TABLES, CV_SCORES, OOF_DIR, PREDICTIONS_DIR, MODELS_DIR, FIGURES, DELIVERABLES, CONFIGS]:
    path.mkdir(parents=True, exist_ok=True)

# ── Constants ──────────────────────────────────────────────────────────
ID_COLUMN = "Id"

TARGET_COLUMNS = [
    "Q0_TOTAL_ASSETS", "Q0_TOTAL_LIABILITIES", "Q0_TOTAL_STOCKHOLDERS_EQUITY",
    "Q0_GROSS_PROFIT", "Q0_COST_OF_REVENUES", "Q0_REVENUES",
    "Q0_OPERATING_INCOME", "Q0_OPERATING_EXPENSES", "Q0_EBITDA",
]

SUBMISSION_COLUMNS = [
    "Id", "Q0_REVENUES", "Q0_COST_OF_REVENUES", "Q0_GROSS_PROFIT",
    "Q0_OPERATING_EXPENSES", "Q0_EBITDA", "Q0_OPERATING_INCOME",
    "Q0_TOTAL_ASSETS", "Q0_TOTAL_LIABILITIES", "Q0_TOTAL_STOCKHOLDERS_EQUITY",
]

METADATA_COLUMNS = [
    "industry", "sector", "fullTimeEmployees", "auditRisk", "boardRisk",
    "compensationRisk", "shareHolderRightsRisk", "overallRisk",
    "trailingPE", "forwardPE", "floatShares", "sharesOutstanding",
    "trailingEps", "forwardEps", "targetHighPrice", "targetLowPrice",
    "targetMeanPrice", "targetMedianPrice", "recommendationMean",
    "recommendationKey", "numberOfAnalystOpinions", "totalCash",
    "totalCashPerShare", "ebitda", "totalDebt", "totalRevenue",
    "revenuePerShare", "freeCashflow", "operatingCashflow",
    "revenueGrowth", "financialCurrency",
]

HISTORICAL_METRICS = [
    "TOTAL_ASSETS", "TOTAL_CURRENT_ASSETS", "TOTAL_NONCURRENT_ASSETS",
    "TOTAL_LIABILITIES", "TOTAL_CURRENT_LIABILITIES", "TOTAL_NONCURRENT_LIABILITIES",
    "TOTAL_LIABILITIES_AND_EQUITY", "TOTAL_STOCKHOLDERS_EQUITY",
    "NET_INCOME", "GROSS_PROFIT", "COST_OF_REVENUES", "REVENUES",
    "OPERATING_INCOME", "OPERATING_EXPENSES", "EBITDA",
    "DEPRECIATION_AND_AMORTIZATION",
]

HISTORICAL_QUARTERS = [f"Q{i}" for i in range(1, 11)]

QUARTER_VALUE_PATTERN = re.compile(r"^Q([0-9]+)_([A-Z0-9_]+)$")
QUARTER_FYE_PATTERN = re.compile(r"^Q([0-9]+)_fiscal_year_end$")

# ── Plot style ─────────────────────────────────────────────────────────
plt.style.use("seaborn-v0_8-whitegrid")
plt.rcParams.update({
    "figure.dpi": 160, "savefig.dpi": 160,
    "font.sans-serif": ["Microsoft YaHei", "SimHei", "DejaVu Sans"],
    "axes.unicode_minus": False,
    "axes.titlesize": 11, "axes.labelsize": 10, "legend.fontsize": 9,
})

print("✓ 常量和路径初始化完成")
print(f"  ROOT = {ROOT}")

### 0.2 工具函数

In [ ]:
# ═══════════════════════════════════════════════════════════════════════
# Utility functions (all inline — no src/ dependencies)
# ═══════════════════════════════════════════════════════════════════════

def replace_inf_with_nan(frame: pd.DataFrame) -> pd.DataFrame:
    return frame.replace([np.inf, -np.inf], np.nan)

def safe_ratio(
    numerator: pd.Series | np.ndarray,
    denominator: pd.Series | np.ndarray,
    eps: float = 1e-9,
):
    num = np.asarray(numerator, dtype="float64")
    den = np.asarray(denominator, dtype="float64")
    shape = np.broadcast_shapes(num.shape, den.shape)
    out = np.full(shape, np.nan, dtype="float64")
    num = np.broadcast_to(num, shape)
    den = np.broadcast_to(den, shape)
    valid = np.abs(den) >= eps
    np.divide(num, den, out=out, where=valid)
    return out

def signed_log1p(values: pd.Series | np.ndarray):
    arr = np.asarray(values, dtype="float64")
    return np.sign(arr) * np.log1p(np.abs(arr))

def stable_row_hash(frame: pd.DataFrame) -> pd.Series:
    ordered = frame.sort_index(axis=1)
    return pd.util.hash_pandas_object(ordered, index=False)

def r2_by_target(
    y_true: pd.DataFrame, y_pred: pd.DataFrame,
    targets: Sequence[str] = TARGET_COLUMNS,
) -> dict[str, float]:
    scores: dict[str, float] = {}
    for target in targets:
        tv = pd.to_numeric(y_true[target], errors="coerce")
        pv = pd.to_numeric(y_pred[target], errors="coerce")
        valid = tv.notna() & pv.notna()
        scores[target] = float(r2_score(tv.loc[valid], pv.loc[valid])) if valid.sum() >= 2 else math.nan
    return scores

def mean_target_r2(scores: Mapping[str, float]) -> float:
    vals = [v for v in scores.values() if pd.notna(v)]
    return float(np.mean(vals)) if vals else math.nan

def _save_fig(fig: plt.Figure, name: str) -> None:
    path = FIGURES / name
    path.parent.mkdir(parents=True, exist_ok=True)
    fig.tight_layout()
    fig.savefig(path, bbox_inches="tight")
    plt.close(fig)
    print(f"  ✓ 图表已保存: {path}")

def _display_target_name(name: str) -> str:
    mapping = {
        "Q0_TOTAL_ASSETS": "总资产", "Q0_TOTAL_LIABILITIES": "总负债",
        "Q0_TOTAL_STOCKHOLDERS_EQUITY": "股东权益", "Q0_GROSS_PROFIT": "毛利",
        "Q0_COST_OF_REVENUES": "营业成本", "Q0_REVENUES": "营业收入",
        "Q0_OPERATING_INCOME": "营业利润", "Q0_OPERATING_EXPENSES": "营业费用",
        "Q0_EBITDA": "EBITDA",
    }
    return mapping.get(name, name)

def _display_experiment_name(name: str) -> str:
    mapping = {
        "B0": "B0 均值", "B1": "B1 近一期", "B2": "B2 上年同期",
        "B3": "B3 趋势外推", "B4": "B4 规则融合",
        "M1_ridge": "M1 Ridge", "M3d_catboost_direct": "M3d CatBoost直模",
        "M4_catboost_residual": "M4 CatBoost残差",
        "M6_oof_blend": "M6 OOF融合", "M7_accounting_postprocess": "M7 会计后处理",
    }
    return mapping.get(name, name)

print("✓ 工具函数定义完成")

## 1. 数据读取与结构检查

In [ ]:
# ═══════════════════════════════════════════════════════════════════════
# 1. 数据读取与结构检查
# ═══════════════════════════════════════════════════════════════════════

print("=" * 60)
print("1. 数据读取与结构检查")
print("=" * 60)

train = pd.read_csv(ROOT / "train.csv")
test = pd.read_csv(ROOT / "test.csv")
sample_submission = pd.read_csv(ROOT / "sample_submission.csv")

print(f"train 形状:        {train.shape}")
print(f"test 形状:         {test.shape}")
print(f"sample_submission: {sample_submission.shape}")

# Schema validation
assert train.shape == (1624, 212), f"train shape mismatch: {train.shape}"
assert test.shape == (406, 203), f"test shape mismatch: {test.shape}"
assert sample_submission.shape == (406, 10), f"sample_submission shape mismatch: {sample_submission.shape}"

missing_targets = [c for c in TARGET_COLUMNS if c not in train.columns]
assert not missing_targets, f"Missing targets in train: {missing_targets}"
assert list(sample_submission.columns) == SUBMISSION_COLUMNS, "submission columns mismatch"

# Replace inf
train = replace_inf_with_nan(train)
test = replace_inf_with_nan(test)

# ID check
assert train[ID_COLUMN].dtype == test[ID_COLUMN].dtype, "ID dtype mismatch"
assert train[ID_COLUMN].nunique() == len(train), "train ID not unique"

print(f"\n✓ Schema 验证通过")
print(f"  train 样本数:     {len(train)}")
print(f"  test 样本数:      {len(test)}")
print(f"  特征列 (train):   {train.shape[1] - 1} (含ID)")
print(f"  目标列:           {len(TARGET_COLUMNS)}")

# Column categorization
cols_info = {
    "id": [c for c in [ID_COLUMN] if c in train.columns],
    "targets": [c for c in TARGET_COLUMNS if c in train.columns],
    "metadata": [c for c in METADATA_COLUMNS if c in train.columns],
    "historical": [c for c in train.columns if QUARTER_VALUE_PATTERN.match(c) or QUARTER_FYE_PATTERN.match(c)],
}
cols_info["other"] = [c for c in train.columns if c not in set(sum(cols_info.values(), []))]
print(f"\n  列分类:")
for k, v in cols_info.items():
    print(f"    {k}: {len(v)} 列")

## 2. EDA — 数据质量分析

In [ ]:
# ═══════════════════════════════════════════════════════════════════════
# 2. EDA — 缺失值、异常值、重复值分析
# ═══════════════════════════════════════════════════════════════════════

print("=" * 60)
print("2. EDA — 数据质量分析")
print("=" * 60)

# ── 2.1 缺失值 ────────────────────────────────────────────────────────
model_cols = [c for c in train.columns if c not in {ID_COLUMN, *TARGET_COLUMNS}]
missing_rates = train[model_cols].isna().mean().sort_values(ascending=False)
print(f"\n2.1 缺失值分析")
print(f"  总特征数: {len(model_cols)}")
print(f"  缺失率 > 50% 的特征: {int((missing_rates > 0.5).sum())}")
print(f"  缺失率 > 80% 的特征: {int((missing_rates > 0.8).sum())}")
print(f"  完全缺失的特征: {int((missing_rates == 1.0).sum())}")
print(f"\n  缺失率最高的10个特征:")
for col in missing_rates.head(10).index:
    print(f"    {col}: {missing_rates[col]:.2%}")

# 缺失率按季度
quarter_missing = []
for q in HISTORICAL_QUARTERS:
    q_cols = [c for c in model_cols if c.startswith(f"{q}_")]
    if q_cols:
        quarter_missing.append({"quarter": q, "avg_missing_rate": train[q_cols].isna().mean().mean()})
quarter_missing_df = pd.DataFrame(quarter_missing)
print(f"\n  各季度平均缺失率:")
for _, row in quarter_missing_df.iterrows():
    print(f"    {row['quarter']}: {row['avg_missing_rate']:.2%}")

# ── 2.2 行缺失 ────────────────────────────────────────────────────────
row_missing = train[model_cols].isna().sum(axis=1)
print(f"\n2.2 行级缺失")
print(f"  每行平均缺失列数: {row_missing.mean():.1f}")
print(f"  每行中位缺失列数: {row_missing.median():.1f}")
print(f"  行缺失率均值:     {train[model_cols].isna().mean(axis=1).mean():.2%}")

# ── 2.3 重复行 ────────────────────────────────────────────────────────
dup_rows = train[model_cols].duplicated().sum()
print(f"\n2.3 重复行")
print(f"  完全重复的特征行: {dup_rows}")

# ── 2.4 目标变量 ──────────────────────────────────────────────────────
print(f"\n2.4 目标变量统计")
for target in TARGET_COLUMNS:
    vals = train[target].astype(float).dropna()
    print(f"  {target}: "
          f"mean={vals.mean():.2e}, std={vals.std():.2e}, "
          f"min={vals.min():.2e}, max={vals.max():.2e}, "
          f"missing={train[target].isna().sum()}")

# ── 2.5 sector 分布 ───────────────────────────────────────────────────
sector_counts = train["sector"].fillna("Missing").value_counts()
print(f"\n2.5 sector 分布 ({len(sector_counts)} 类)")
for s in sector_counts.head(10).index:
    print(f"  {s}: {sector_counts[s]}")

# ── 2.6 目标变量相关性 ────────────────────────────────────────────────
target_corr = train[TARGET_COLUMNS].astype(float).corr()
print(f"\n2.6 目标变量相关性矩阵:")
print(target_corr.to_string(float_format=lambda x: f"{x:.3f}"))

# ── 2.7 会计恒等式检查 ────────────────────────────────────────────────
assets = train["Q0_TOTAL_ASSETS"].astype(float)
liabilities = train["Q0_TOTAL_LIABILITIES"].astype(float)
equity = train["Q0_TOTAL_STOCKHOLDERS_EQUITY"].astype(float)
identity_error = (assets - liabilities - equity) / assets.abs().replace(0, np.nan)
print(f"\n2.7 会计恒等式 (Assets = Liabilities + Equity)")
print(f"  平均绝对相对误差: {identity_error.abs().mean():.6f}")
print(f"  中位绝对相对误差: {identity_error.abs().median():.6f}")
print(f"  误差 > 1% 的样本:  {int((identity_error.abs() > 0.01).sum())} / {len(identity_error)}")

print("\n✓ EDA 基本分析完成")

## 3. EDA — 图表

In [ ]:
# ═══════════════════════════════════════════════════════════════════════
# 3. EDA 图表
# ═══════════════════════════════════════════════════════════════════════

print("=" * 60)
print("3. EDA 图表")
print("=" * 60)

# ── 3.1 sector 分布 ───────────────────────────────────────────────────
fig, ax = plt.subplots(figsize=(10, 5))
sector_counts = train["sector"].fillna("Missing").value_counts().sort_values(ascending=False)
sector_counts.plot(kind="bar", ax=ax, color="#4C78A8")
ax.set_title("行业板块样本分布 (sector)")
ax.set_xlabel("sector"); ax.set_ylabel("样本数")
ax.tick_params(axis="x", rotation=45)
_save_fig(fig, "fig01_sector_distribution.png")

fig, ax = plt.subplots(figsize=(10, 7))
industry_counts = train["industry"].fillna("Missing").value_counts().head(20).sort_values(ascending=True)
industry_counts.plot(kind="barh", ax=ax, color="#F58518")
ax.set_title("行业 Top 20 样本分布")
ax.set_xlabel("样本数"); ax.set_ylabel("industry")
_save_fig(fig, "fig02_industry_top20.png")

# ── 3.2 缺失率 Top20 ──────────────────────────────────────────────────
fig, ax = plt.subplots(figsize=(10, 7))
top_missing = missing_rates.sort_values(ascending=False).head(20).sort_values(ascending=True)
top_missing.plot(kind="barh", ax=ax, color="#E45756")
ax.set_title("缺失率最高的20个字段")
ax.set_xlabel("缺失率"); ax.set_ylabel("字段")
_save_fig(fig, "fig03_missing_top20.png")

# ── 3.3 各季度缺失率 ──────────────────────────────────────────────────
fig, ax = plt.subplots(figsize=(10, 4.5))
ax.plot(quarter_missing_df["quarter"], quarter_missing_df["avg_missing_rate"],
        marker="o", color="#72B7B2")
ax.set_title("各季度平均缺失率")
ax.set_xlabel("季度"); ax.set_ylabel("平均缺失率")
_save_fig(fig, "fig04_missing_by_quarter.png")

# ── 3.4 目标变量 signed-log 分布 ──────────────────────────────────────
ncols, nrows = 3, int(np.ceil(len(TARGET_COLUMNS) / 3))
fig, axes = plt.subplots(nrows, ncols, figsize=(14, 10))
axes = np.asarray(axes).reshape(-1)
for ax, col in zip(axes, TARGET_COLUMNS):
    vals_log = signed_log1p(train[col].astype(float))
    vals_clean = vals_log[~np.isnan(vals_log)]
    ax.hist(vals_clean, bins=40, color="#54A24B", alpha=0.85)
    ax.set_title(_display_target_name(col)); ax.set_xlabel("signed log1p")
for ax in axes[len(TARGET_COLUMNS):]:
    ax.axis("off")
fig.suptitle("9个目标的 signed-log 分布", y=1.01, fontsize=13)
_save_fig(fig, "fig05_target_distributions.png")

lag_corr_rows = []
for target in TARGET_COLUMNS:
    metric = target.removeprefix("Q0_")
    row = {"target": target}
    for q in HISTORICAL_QUARTERS:
        col = f"{q}_{metric}"
        if col in train.columns:
            mask = train[target].notna() & train[col].notna()
            row[q] = train.loc[mask, target].astype(float).corr(train.loc[mask, col].astype(float)) if mask.sum() >= 2 else np.nan
        else:
            row[q] = np.nan
    lag_corr_rows.append(row)
lag_corr_df = pd.DataFrame(lag_corr_rows).set_index("target")
fig, ax = plt.subplots(figsize=(12, 5.6))
im = ax.imshow(lag_corr_df.values, aspect="auto", cmap="RdYlGn", vmin=-1, vmax=1)
ax.set_xticks(range(len(lag_corr_df.columns)))
ax.set_xticklabels(lag_corr_df.columns, rotation=35, ha="right")
ax.set_yticks(range(len(lag_corr_df.index)))
ax.set_yticklabels([_display_target_name(c) for c in lag_corr_df.index])
ax.set_title("鍘嗗彶鍚庤繍涓庣洰鏍囩殑鐩稿叧鎬х儹鍔涘浘")
fig.colorbar(im, ax=ax, fraction=0.02, pad=0.02)
_save_fig(fig, "fig06_lag_correlation_heatmap.png")

# ── 3.5 目标变量相关性热力图 ──────────────────────────────────────────
fig, ax = plt.subplots(figsize=(8.5, 7))
im = ax.imshow(target_corr.values, cmap="RdBu_r", vmin=-1, vmax=1)
ax.set_xticks(range(len(target_corr.columns)))
ax.set_xticklabels([_display_target_name(c) for c in target_corr.columns], rotation=35, ha="right")
ax.set_yticks(range(len(target_corr.index)))
ax.set_yticklabels([_display_target_name(c) for c in target_corr.index])
ax.set_title("目标变量相关性热力图")
fig.colorbar(im, ax=ax, fraction=0.03, pad=0.02)
_save_fig(fig, "fig08_target_correlation_heatmap.png")

# ── 3.6 会计恒等式误差 ────────────────────────────────────────────────
fig, ax = plt.subplots(figsize=(10, 4.5))
ax.hist(identity_error.dropna(), bins=60, color="#B279A2", alpha=0.85)
ax.set_title("会计恒等式相对误差分布\n(Assets - Liabilities - Equity) / |Assets|")
ax.set_xlabel("相对误差"); ax.set_ylabel("样本数")
_save_fig(fig, "fig07_accounting_identity_error.png")

print("\n✓ EDA 图表生成完成")

## 4. 特征工程

特征工程逻辑与 `src/feature_engineering.py` 完全一致，包含：
- 原始历史数据 (Q1–Q10 的16个财务指标 + fiscal_year_end)
- 行业/板块 (industry, sector)
- 元数据 (估值、风险、分析师等 29 个字段)
- 工程特征：时序统计 (均值/中位/标准差/斜率)、环比变化、财务比率、缺失率
- signed-log 变换 (对高量纲特征)

In [ ]:
# ═══════════════════════════════════════════════════════════════════════
# 4. 特征工程 (所有函数内联，与 src/feature_engineering.py 逻辑一致)
# ═══════════════════════════════════════════════════════════════════════

print("=" * 60)
print("4. 特征工程")
print("=" * 60)

# ── Helper functions ───────────────────────────────────────────────────
def metric_from_target(target: str) -> str:
    return target.removeprefix("Q0_")

def historical_columns_for_target(target: str) -> list[str]:
    metric = metric_from_target(target)
    return [f"{q}_{metric}" for q in HISTORICAL_QUARTERS]

def historical_value_columns(frame: pd.DataFrame) -> list[str]:
    cols = []
    for q in HISTORICAL_QUARTERS:
        for m in HISTORICAL_METRICS:
            c = f"{q}_{m}"
            if c in frame.columns:
                cols.append(c)
    return cols

def historical_fiscal_columns(frame: pd.DataFrame) -> list[str]:
    cols = ["Q0_fiscal_year_end"] if "Q0_fiscal_year_end" in frame.columns else []
    for q in HISTORICAL_QUARTERS:
        c = f"{q}_fiscal_year_end"
        if c in frame.columns:
            cols.append(c)
    return cols

def raw_history_feature_columns(frame: pd.DataFrame) -> list[str]:
    return [*historical_fiscal_columns(frame), *historical_value_columns(frame)]

def industry_feature_columns(frame: pd.DataFrame) -> list[str]:
    return [c for c in ["industry", "sector"] if c in frame.columns]

def metadata_feature_columns(frame: pd.DataFrame) -> list[str]:
    return [c for c in METADATA_COLUMNS if c in frame.columns]

def historical_matrix(frame: pd.DataFrame, target: str) -> pd.DataFrame:
    cols = historical_columns_for_target(target)
    missing = [c for c in cols if c not in frame.columns]
    if missing:
        raise KeyError(f"Missing historical columns: {missing}")
    return replace_inf_with_nan(frame.loc[:, cols]).apply(pd.to_numeric, errors="coerce")

def model_feature_columns(frame: pd.DataFrame) -> list[str]:
    return [c for c in frame.columns if c not in {ID_COLUMN, *TARGET_COLUMNS}]

def _metric_history(frame: pd.DataFrame, metric: str) -> pd.DataFrame:
    cols = [f"{q}_{metric}" for q in HISTORICAL_QUARTERS if f"{q}_{metric}" in frame.columns]
    return replace_inf_with_nan(frame.loc[:, cols]).apply(pd.to_numeric, errors="coerce")

def _row_slope(values: np.ndarray, max_points: int) -> np.ndarray:
    out = np.full(values.shape[0], np.nan, dtype="float64")
    lags = np.arange(1, values.shape[1] + 1, dtype="float64")
    for i, row in enumerate(values):
        valid = np.isfinite(row)
        if valid.sum() < 2:
            continue
        x, y = lags[valid][:max_points], row[valid][:max_points]
        if len(y) < 2:
            continue
        slope, _ = np.polyfit(x, y, deg=1)
        out[i] = slope
    return out

def _add_time_series_features(frame: pd.DataFrame, features: dict) -> None:
    for metric in HISTORICAL_METRICS:
        hist = _metric_history(frame, metric)
        if hist.empty:
            continue
        pfx = f"hist_{metric}"
        features[f"{pfx}_last_available"] = hist.bfill(axis=1).iloc[:, 0]
        for n in [2, 4, 8]:
            sub = hist.iloc[:, :n]
            features[f"{pfx}_mean_last_{n}"] = sub.mean(axis=1)
            features[f"{pfx}_non_missing_last_{n}"] = sub.notna().sum(axis=1)
        features[f"{pfx}_median_last_4"] = hist.iloc[:, :4].median(axis=1)
        features[f"{pfx}_std_last_4"] = hist.iloc[:, :4].std(axis=1, ddof=0)
        features[f"{pfx}_std_last_8"] = hist.iloc[:, :8].std(axis=1, ddof=0)
        features[f"{pfx}_min_last_4"] = hist.iloc[:, :4].min(axis=1)
        features[f"{pfx}_max_last_4"] = hist.iloc[:, :4].max(axis=1)
        features[f"{pfx}_slope_last_4"] = _row_slope(hist.iloc[:, :4].to_numpy(dtype="float64"), 4)
        features[f"{pfx}_slope_last_8"] = _row_slope(hist.iloc[:, :8].to_numpy(dtype="float64"), 8)
        features[f"{pfx}_non_missing_count"] = hist.notna().sum(axis=1)
        features[f"{pfx}_missing_count"] = hist.isna().sum(axis=1)

def _add_change_features(frame: pd.DataFrame, features: dict) -> None:
    for metric in HISTORICAL_METRICS:
        q1c, q2c, q3c, q5c = f"Q1_{metric}", f"Q2_{metric}", f"Q3_{metric}", f"Q5_{metric}"
        if q1c not in frame.columns or q2c not in frame.columns:
            continue
        q1 = pd.to_numeric(frame[q1c], errors="coerce")
        q2 = pd.to_numeric(frame[q2c], errors="coerce")
        q3 = pd.to_numeric(frame[q3c], errors="coerce") if q3c in frame.columns else None
        q5 = pd.to_numeric(frame[q5c], errors="coerce") if q5c in frame.columns else None
        pfx = f"chg_{metric}"
        features[f"{pfx}_q1_minus_q2"] = q1 - q2
        if q3 is not None:
            features[f"{pfx}_q2_minus_q3"] = q2 - q3
        if q5 is not None:
            features[f"{pfx}_q1_minus_q5"] = q1 - q5
        features[f"{pfx}_q1_over_q2_minus_1"] = safe_ratio(q1, q2) - 1
        if q3 is not None:
            features[f"{pfx}_q2_over_q3_minus_1"] = safe_ratio(q2, q3) - 1
        if q5 is not None:
            features[f"{pfx}_q1_over_q5_minus_1"] = safe_ratio(q1, q5) - 1

def _add_financial_ratio_features(frame: pd.DataFrame, features: dict) -> None:
    specs = {
        "debt_ratio": ("TOTAL_LIABILITIES", "TOTAL_ASSETS"),
        "equity_ratio": ("TOTAL_STOCKHOLDERS_EQUITY", "TOTAL_ASSETS"),
        "current_ratio": ("TOTAL_CURRENT_ASSETS", "TOTAL_CURRENT_LIABILITIES"),
        "noncurrent_assets_ratio": ("TOTAL_NONCURRENT_ASSETS", "TOTAL_ASSETS"),
        "gross_margin": ("GROSS_PROFIT", "REVENUES"),
        "operating_margin": ("OPERATING_INCOME", "REVENUES"),
        "ebitda_margin": ("EBITDA", "REVENUES"),
        "cost_rate": ("COST_OF_REVENUES", "REVENUES"),
        "expense_rate": ("OPERATING_EXPENSES", "REVENUES"),
    }
    for quarter in ["Q1", "Q2", "Q4", "Q5"]:
        for rname, (num, denom) in specs.items():
            nc, dc = f"{quarter}_{num}", f"{quarter}_{denom}"
            if nc not in frame.columns or dc not in frame.columns:
                continue
            features[f"{quarter}_{rname}"] = safe_ratio(
                pd.to_numeric(frame[nc], errors="coerce"),
                pd.to_numeric(frame[dc], errors="coerce"),
            )

def _add_missing_features(frame: pd.DataFrame, features: dict) -> None:
    history_cols = raw_history_feature_columns(frame)
    meta_cols = [c for c in METADATA_COLUMNS if c in frame.columns]
    model_cols_local = [c for c in model_feature_columns(frame) if c in frame.columns]
    features["row_missing_count"] = frame[model_cols_local].isna().sum(axis=1)
    features["row_missing_ratio"] = frame[model_cols_local].isna().mean(axis=1)
    features["history_missing_count"] = frame[history_cols].isna().sum(axis=1)
    features["metadata_missing_count"] = frame[meta_cols].isna().sum(axis=1) if meta_cols else 0
    q_avail = []
    for q in HISTORICAL_QUARTERS:
        q_cols = [f"{q}_{m}" for m in HISTORICAL_METRICS if f"{q}_{m}" in frame.columns]
        if q_cols:
            q_avail.append(frame[q_cols].notna().any(axis=1).astype(int))
    features["quarter_available_count"] = sum(q_avail) if q_avail else 0
    important = ["Q1_TOTAL_ASSETS", "Q1_TOTAL_LIABILITIES", "Q1_TOTAL_STOCKHOLDERS_EQUITY",
                 "Q1_REVENUES", "Q1_EBITDA", "trailingPE"]
    for col in important:
        if col in frame.columns:
            features[f"is_missing_{col}"] = frame[col].isna().astype(int)

def _signed_log_feature_frame(features: pd.DataFrame) -> pd.DataFrame:
    generated: dict = {}
    for col in [c for c in features.columns if pd.api.types.is_numeric_dtype(features[c])]:
        if col.endswith("_fiscal_year_end") or col.startswith("is_missing_"):
            continue
        s = pd.to_numeric(features[col], errors="coerce")
        if s.notna().sum() == 0:
            continue
        if s.abs().quantile(0.95) <= 10:
            continue
        generated[f"slog_{col}"] = signed_log1p(s)
    return pd.DataFrame(generated, index=features.index)

def build_feature_frame(frame: pd.DataFrame, feature_set: str = "history_raw") -> pd.DataFrame:
    """Build deterministic features -- no fitting on validation or test data."""
    source = replace_inf_with_nan(frame.copy())
    allowed = {"history_raw", "history_industry", "history_metadata",
                "history_engineered", "history_metadata_engineered"}
    if feature_set not in allowed:
        raise ValueError(f"Unknown feature_set: {feature_set}")
    base_cols = raw_history_feature_columns(source)
    if feature_set == "history_industry":
        base_cols = [*base_cols, *industry_feature_columns(source)]
    elif feature_set in {"history_metadata", "history_metadata_engineered"}:
        base_cols = [*base_cols, *metadata_feature_columns(source)]
    features = source.loc[:, base_cols].copy()
    if feature_set in {"history_engineered", "history_metadata_engineered"}:
        generated: dict = {}
        _add_time_series_features(source, generated)
        _add_change_features(source, generated)
        _add_financial_ratio_features(source, generated)
        _add_missing_features(source, generated)
        if generated:
            features = pd.concat([features, pd.DataFrame(generated, index=source.index)], axis=1)
        slog = _signed_log_feature_frame(features)
        if not slog.empty:
            features = pd.concat([features, slog], axis=1)
    features = replace_inf_with_nan(features)
    leaked = [c for c in TARGET_COLUMNS if c in features.columns]
    if leaked:
        raise AssertionError(f"Q0 target columns leaked: {leaked}")
    return features

def build_feature_group_id(frame: pd.DataFrame, feature_cols: Sequence[str] | None = None) -> pd.Series:
    if feature_cols is None:
        feature_cols = [c for c in frame.columns if c not in {ID_COLUMN, *TARGET_COLUMNS}]
    feats = replace_inf_with_nan(frame.loc[:, list(feature_cols)].copy())
    hashes = stable_row_hash(feats)
    labels = pd.factorize(hashes, sort=True)[0]
    return pd.Series(labels, index=frame.index, name="group_id")

# ── Build feature frames ───────────────────────────────────────────────
X_train_raw = build_feature_frame(train, feature_set="history_raw")
X_test_raw = build_feature_frame(test, feature_set="history_raw")
X_train_full = build_feature_frame(train, feature_set="history_metadata_engineered")
X_test_full = build_feature_frame(test, feature_set="history_metadata_engineered")

print(f"  history_raw 特征数:          {X_train_raw.shape[1]}")
print(f"  history_metadata_engineered: {X_train_full.shape[1]}")

y = train[TARGET_COLUMNS].astype(float)
print(f"\n✓ 特征工程完成")

## 5. GroupKFold 交叉验证设置

使用基于特征哈希的 GroupKFold (n_splits=5)，确保同一特征模式的样本不跨折泄漏。

In [ ]:
# ═══════════════════════════════════════════════════════════════════════
# 5. 交叉验证设置 (GroupKFold, n_splits=5)
# ═══════════════════════════════════════════════════════════════════════

print("=" * 60)
print("5. GroupKFold 交叉验证设置")
print("=" * 60)

def make_groupkfold_splits(groups, n_splits: int = 5) -> list[tuple[np.ndarray, np.ndarray]]:
    groups_arr = np.asarray(groups)
    unique = np.unique(groups_arr)
    assert len(unique) >= n_splits, f"Need {n_splits} unique groups, got {len(unique)}"
    splitter = GroupKFold(n_splits=n_splits)
    return [(t, v) for t, v in splitter.split(np.zeros((len(groups_arr), 1)), groups=groups_arr)]

def assert_no_group_leakage(splits, groups) -> None:
    groups_arr = np.asarray(groups)
    for fold, (ti, vi) in enumerate(splits):
        leaked = set(groups_arr[ti]) & set(groups_arr[vi])
        assert not leaked, f"Fold {fold} leaks {len(leaked)} groups"

groups = build_feature_group_id(train, feature_cols=model_feature_columns(train))
splits = make_groupkfold_splits(groups, n_splits=5)
assert_no_group_leakage(splits, groups)

unique_groups = len(np.unique(groups))
print(f"  唯一特征组数: {unique_groups}")
print(f"  折数:          5")
for fold, (ti, vi) in enumerate(splits):
    print(f"  Fold {fold}: train={len(ti)}, valid={len(vi)}")

print("\n✓ GroupKFold 设置完成")

## 6. 基线模型 B0–B4

- **B0:** 训练集均值
- **B1:** 最近一期复制 (Q1 → Q0)
- **B2:** 上年同期复制 (Q4 → Q0, 季节性)
- **B3:** 线性趋势外推
- **B4:** 逐目标 OOF 加权融合 B1+B2+B3

In [ ]:
# ═══════════════════════════════════════════════════════════════════════
# 6. 基线模型 B0–B4 (规则方法 + OOF 融合)
# ═══════════════════════════════════════════════════════════════════════

print("=" * 60)
print("6. 基线模型 B0–B4")
print("=" * 60)

# ── Baseline rule functions ────────────────────────────────────────────
def predict_recent_copy(frame: pd.DataFrame, target: str, fallback: float) -> pd.Series:
    history = historical_matrix(frame, target)
    ordered = historical_columns_for_target(target)
    out = history.loc[:, ordered].bfill(axis=1).iloc[:, 0]
    return out.fillna(fallback).astype(float)

def predict_seasonal_copy(frame: pd.DataFrame, target: str, fallback: float) -> pd.Series:
    history = historical_matrix(frame, target)
    metric = metric_from_target(target)
    order = ["Q4", "Q3", "Q5", "Q2", "Q6", "Q1", "Q7", "Q8", "Q9", "Q10"]
    cols = [f"{q}_{metric}" for q in order]
    out = history.loc[:, cols].bfill(axis=1).iloc[:, 0]
    return out.fillna(fallback).astype(float)

def predict_short_trend(frame: pd.DataFrame, target: str, fallback: float, max_pts: int = 4) -> pd.Series:
    history = historical_matrix(frame, target)
    vals = history.to_numpy(dtype="float64")
    preds = np.full(vals.shape[0], float(fallback), dtype="float64")
    lags = np.arange(1, len(HISTORICAL_QUARTERS) + 1, dtype="float64")
    for i, row in enumerate(vals):
        valid = np.isfinite(row)
        if valid.sum() == 0:
            continue
        cl, cv = lags[valid][:max_pts], row[valid][:max_pts]
        if len(cv) == 1:
            preds[i] = cv[0]
        else:
            slope, intercept = np.polyfit(cl, cv, deg=1)
            if np.isfinite(intercept):
                preds[i] = intercept
    return pd.Series(preds, index=frame.index, name=target)

def predict_rule(rule_id: str, frame: pd.DataFrame, target: str,
                 train_target_vals: pd.Series) -> pd.Series:
    fb = float(train_target_vals.median())
    if rule_id == "B0":
        return pd.Series(float(train_target_vals.mean()), index=frame.index, name=target)
    if rule_id == "B1":
        return predict_recent_copy(frame, target, fb)
    if rule_id == "B2":
        return predict_seasonal_copy(frame, target, fb)
    if rule_id == "B3":
        return predict_short_trend(frame, target, fb)
    raise ValueError(f"Unknown rule: {rule_id}")

def simplex_weights(n: int, step: float = 0.1) -> list[tuple]:
    units = int(round(1.0 / step))
    if not math.isclose(units * step, 1.0, rel_tol=0.0, abs_tol=1e-12):
        raise ValueError("step must divide 1.0")
    result = []
    for parts in itertools.product(range(units + 1), repeat=n):
        if sum(parts) == units:
            result.append(tuple(p / units for p in parts))
    return result

def blend_predictions(predictions: Mapping, target: str, members: Sequence,
                      weights: Sequence) -> pd.Series:
    out = None
    for m, w in zip(members, weights):
        s = predictions[m][target].astype(float) * float(w)
        out = s.copy() if out is None else out + s
    return out.rename(target)

def masked_r2(y_true: pd.Series, y_pred: pd.Series) -> float:
    tv = pd.to_numeric(y_true, errors="coerce")
    pv = pd.to_numeric(y_pred, errors="coerce")
    valid = tv.notna() & pv.notna()
    return float(r2_score(tv.loc[valid], pv.loc[valid])) if valid.sum() >= 2 else math.nan

# ── Run OOF baselines ──────────────────────────────────────────────────
t0 = time.perf_counter()
y_data = train[TARGET_COLUMNS].astype(float)
fold_assignments = pd.Series(index=train.index, dtype="int64")

# OOF predictions for B0-B3
base_preds = {r: pd.DataFrame(index=train.index, columns=TARGET_COLUMNS, dtype="float64")
              for r in ["B0", "B1", "B2", "B3"]}

for fold, (train_idx, valid_idx) in enumerate(splits):
    train_part = train.iloc[train_idx]
    valid_part = train.iloc[valid_idx]
    y_train_fold = train_part[TARGET_COLUMNS].astype(float)
    fold_assignments.iloc[valid_idx] = fold
    for rule in ["B0", "B1", "B2", "B3"]:
        for target in TARGET_COLUMNS:
            base_preds[rule].loc[valid_part.index, target] = \
                predict_rule(rule, valid_part, target, y_train_fold[target])

# Compute B0-B3 OOF scores
baseline_scores = []
for rule in ["B0", "B1", "B2", "B3"]:
    scores = r2_by_target(y_data, base_preds[rule])
    row = {"experiment_id": rule, "model_name": f"{rule}_baseline",
           "mean_r2": mean_target_r2(scores)}
    row.update({f"r2_{t}": s for t, s in scores.items()})
    baseline_scores.append(row)
    print(f"  {rule}: mean_r2={row['mean_r2']:.4f}")

# ── Search B4 blend weights on OOF ─────────────────────────────────────
candidate_sets = [
    ("B1", ("B1",)), ("B2", ("B2",)), ("B3", ("B3",)),
    ("B1+B2", ("B1", "B2")), ("B1+B3", ("B1", "B3")),
    ("B1+B2+B3", ("B1", "B2", "B3")),
]
w_by_size = {s: simplex_weights(s, step=0.1) for s in (1, 2, 3)}
b4_config = {}
for target in TARGET_COLUMNS:
    best_score = -math.inf
    best_members, best_w = None, None
    for cname, members in candidate_sets:
        for w in w_by_size[len(members)]:
            pred = blend_predictions(base_preds, target, members, w)
            score = masked_r2(y_data[target], pred)
            if score > best_score:
                best_score, best_members, best_w = score, members, w
    b4_config[target] = {"members": list(best_members),
                         "weights": [float(x) for x in best_w], "oof_r2": float(best_score)}
    print(f"  B4 {_display_target_name(target)}: members={best_members}, "
          f"weights={[f'{x:.1f}' for x in best_w]}, oof_r2={best_score:.4f}")

# Build B4 OOF
b4_pred = pd.DataFrame(index=train.index, columns=TARGET_COLUMNS, dtype="float64")
for target in TARGET_COLUMNS:
    cfg = b4_config[target]
    b4_pred[target] = blend_predictions(base_preds, target, cfg["members"], cfg["weights"])

b4_scores = r2_by_target(y_data, b4_pred)
b4_row = {"experiment_id": "B4", "model_name": "B4_baseline_blend",
          "mean_r2": mean_target_r2(b4_scores)}
b4_row.update({f"r2_{t}": s for t, s in b4_scores.items()})
baseline_scores.append(b4_row)
print(f"  B4: mean_r2={b4_row['mean_r2']:.4f}")

# Build B4 test predictions
test_base_preds = {}
for rule in ["B0", "B1", "B2", "B3"]:
    test_base_preds[rule] = pd.DataFrame(index=test.index, columns=TARGET_COLUMNS, dtype="float64")
    for target in TARGET_COLUMNS:
        test_base_preds[rule][target] = predict_rule(rule, test, target, y_data[target])

b4_test = pd.DataFrame(index=test.index, columns=TARGET_COLUMNS, dtype="float64")
for target in TARGET_COLUMNS:
    cfg = b4_config[target]
    b4_test[target] = blend_predictions(test_base_preds, target, cfg["members"], cfg["weights"])

t_baseline = time.perf_counter() - t0
print(f"\n✓ 基线模型完成 ({t_baseline:.1f}s)")

## 7. 机器学习对照模型 — Ridge Regression

Ridge 回归作为线性基线，使用原始历史数据 (171 个特征)，通过中位数填充 + RobustScaler + OneHotEncoder 预处理。

In [ ]:
# ═══════════════════════════════════════════════════════════════════════
# 7. 机器学习对照模型 — Ridge Regression (M1)
# ═══════════════════════════════════════════════════════════════════════

print("=" * 60)
print("7. 机器学习基线 — Ridge (M1)")
print("=" * 60)

t0 = time.perf_counter()

def split_feature_columns(frame: pd.DataFrame):
    numeric = frame.select_dtypes(include=[np.number, "bool"]).columns.tolist()
    categorical = [c for c in frame.columns if c not in numeric]
    return {"numeric": numeric, "categorical": categorical}

def make_ridge_pipeline(frame: pd.DataFrame, alpha: float = 10.0) -> Pipeline:
    cols = split_feature_columns(frame)
    transformers = []
    if cols["numeric"]:
        transformers.append(("num", Pipeline([
            ("imputer", SimpleImputer(strategy="median")),
            ("scaler", RobustScaler()),
        ]), cols["numeric"]))
    if cols["categorical"]:
        transformers.append(("cat", Pipeline([
            ("imputer", SimpleImputer(strategy="constant", fill_value="Missing")),
            ("encoder", OneHotEncoder(handle_unknown="ignore", sparse_output=False)),
        ]), cols["categorical"]))
    pre = ColumnTransformer(transformers=transformers, remainder="drop", sparse_threshold=0.0)
    return Pipeline([("preprocess", pre),
                     ("model", Ridge(alpha=alpha, solver="svd", tol=1e-4, random_state=42))])

ridge_oof = pd.DataFrame(index=train.index, columns=TARGET_COLUMNS, dtype="float64")
ridge_test = pd.DataFrame(index=test.index, columns=TARGET_COLUMNS, dtype="float64")
ridge_fold_rows = []

for target in TARGET_COLUMNS:
    for fold, (train_idx, valid_idx) in enumerate(splits):
        est = make_ridge_pipeline(X_train_raw)
        Xt = X_train_raw.iloc[train_idx]; Xv = X_train_raw.iloc[valid_idx]
        yt = y[target].iloc[train_idx]
        mask = yt.notna()
        if mask.sum() < 2:
            continue
        est.fit(Xt.loc[mask], yt.loc[mask])
        pred = pd.Series(est.predict(Xv), index=Xv.index)
        ridge_oof.loc[pred.index, target] = pred.values
        score = r2_score(y[target].iloc[valid_idx].dropna(),
                         pred.reindex(y[target].iloc[valid_idx].dropna().index))
        ridge_fold_rows.append({"fold": fold, "target": target, "r2": float(score)})

    # Full model for test
    full_est = make_ridge_pipeline(X_train_raw)
    mask_full = y[target].notna()
    full_est.fit(X_train_raw.loc[mask_full], y[target].loc[mask_full])
    ridge_test[target] = full_est.predict(X_test_raw)

ridge_oof_scores = r2_by_target(y, ridge_oof)
ridge_mean = mean_target_r2(ridge_oof_scores)
ridge_fold_df = pd.DataFrame(ridge_fold_rows)
ridge_fold_std = float(ridge_fold_df.groupby("fold")["r2"].mean().std(ddof=0))
print(f"  Ridge M1: mean_oof_r2={ridge_mean:.4f}, fold_std={ridge_fold_std:.4f}")

t_ridge = time.perf_counter() - t0
print(f"✓ Ridge 完成 ({t_ridge:.1f}s)")

t0 = time.perf_counter()

def make_hgb_pipeline(frame: pd.DataFrame) -> Pipeline:
    cols = split_feature_columns(frame)
    transformers = []
    if cols["numeric"]:
        transformers.append(
            ("num", SimpleImputer(strategy="median"), cols["numeric"])
        )
    if cols["categorical"]:
        transformers.append(
            (
                "cat",
                Pipeline(
                    steps=[
                        ("imputer", SimpleImputer(strategy="constant", fill_value="Missing")),
                        (
                            "encoder",
                            OrdinalEncoder(
                                handle_unknown="use_encoded_value",
                                unknown_value=-1,
                                encoded_missing_value=-1,
                            ),
                        ),
                    ]
                ),
                cols["categorical"],
            )
        )
    preprocess = ColumnTransformer(transformers=transformers, remainder="drop", sparse_threshold=0.0)
    model = HistGradientBoostingRegressor(
        loss="squared_error",
        learning_rate=0.05,
        max_iter=300,
        max_leaf_nodes=31,
        min_samples_leaf=20,
        l2_regularization=0.1,
        early_stopping=True,
        validation_fraction=0.1,
        n_iter_no_change=20,
        random_state=42,
    )
    return Pipeline(steps=[("preprocess", preprocess), ("model", model)])

hgb_oof = pd.DataFrame(index=train.index, columns=TARGET_COLUMNS, dtype="float64")
hgb_test = pd.DataFrame(index=test.index, columns=TARGET_COLUMNS, dtype="float64")
hgb_fold_rows = []

for target in TARGET_COLUMNS:
    for fold, (train_idx, valid_idx) in enumerate(splits):
        est = make_hgb_pipeline(X_train_raw)
        Xt = X_train_raw.iloc[train_idx]
        Xv = X_train_raw.iloc[valid_idx]
        yt = y[target].iloc[train_idx]
        mask = yt.notna()
        if mask.sum() < 2:
            continue
        est.fit(Xt.loc[mask], yt.loc[mask])
        pred = pd.Series(est.predict(Xv), index=Xv.index)
        hgb_oof.loc[pred.index, target] = pred.values
        score = r2_score(y[target].iloc[valid_idx].dropna(), pred.reindex(y[target].iloc[valid_idx].dropna().index))
        hgb_fold_rows.append({"fold": fold, "target": target, "r2": float(score)})

    full_est = make_hgb_pipeline(X_train_raw)
    mask_full = y[target].notna()
    full_est.fit(X_train_raw.loc[mask_full], y[target].loc[mask_full])
    hgb_test[target] = full_est.predict(X_test_raw)

hgb_oof_scores = r2_by_target(y, hgb_oof)
hgb_mean = mean_target_r2(hgb_oof_scores)
hgb_fold_df = pd.DataFrame(hgb_fold_rows)
hgb_fold_std = float(hgb_fold_df.groupby("fold")["r2"].mean().std(ddof=0))
print(f"  HGB M2: mean_oof_r2={hgb_mean:.4f}, fold_std={hgb_fold_std:.4f}")
print(f"✓ HGB 完成 ({time.perf_counter() - t0:.1f}s)")

## 8. CatBoost 数据准备

In [ ]:
# ═══════════════════════════════════════════════════════════════════════
# 8. CatBoost 准备 — 数据预处理
# ═══════════════════════════════════════════════════════════════════════

print("=" * 60)
print("8. CatBoost 数据准备")
print("=" * 60)

def prepare_catboost_frame(frame: pd.DataFrame) -> tuple[pd.DataFrame, list[str]]:
    prepared = frame.copy()
    cols = split_feature_columns(prepared)["categorical"]
    for c in cols:
        prepared[c] = prepared[c].astype("string").fillna("Missing")
    return prepared, cols

CATBOOST_FEATURE_SETS = {
    "history_raw": ("M3a_catboost_direct_history_raw", build_feature_frame(train, feature_set="history_raw"), build_feature_frame(test, feature_set="history_raw")),
    "history_industry": ("M3b_catboost_direct_history_industry", build_feature_frame(train, feature_set="history_industry"), build_feature_frame(test, feature_set="history_industry")),
    "history_metadata": ("M3c_catboost_direct_history_metadata", build_feature_frame(train, feature_set="history_metadata"), build_feature_frame(test, feature_set="history_metadata")),
    "history_metadata_engineered": ("M3d_catboost_direct_history_metadata_engineered", X_train_full, X_test_full),
}

CATBOOST_FRAMES = {}
for feature_set, (experiment_id, train_frame, test_frame) in CATBOOST_FEATURE_SETS.items():
    X_train_cb, cat_features = prepare_catboost_frame(train_frame)
    X_test_cb, _ = prepare_catboost_frame(test_frame)
    CATBOOST_FRAMES[feature_set] = {
        "experiment_id": experiment_id,
        "X_train": X_train_cb,
        "X_test": X_test_cb,
        "cat_features": cat_features,
    }
    print(f"  {feature_set}: {X_train_cb.shape[1]} features, {len(cat_features)} categorical")

# ── Final CatBoost parameters (from configs/best_config) ───────────────
CB_PARAMS = {
    "loss_function": "RMSE",
    "iterations": 180,
    "learning_rate": 0.03,
    "depth": 4,
    "l2_leaf_reg": 8.0,
    "random_seed": 42,
    "verbose": False,
    "allow_writing_files": False,
    "od_type": "Iter",
    "od_wait": 20,
    "use_best_model": True,
}
print(f"\n  CatBoost 最终参数:")
for k, v in CB_PARAMS.items():
    print(f"    {k}: {v}")

## 9. CatBoost 直接预测 (M3a-M3d)

使用 `history_metadata_engineered` 特征集 (所有工程特征)，逐目标独立训练 CatBoost 回归器。

**最终参数:** iterations=180, learning_rate=0.03, depth=4, l2_leaf_reg=8.0, od_wait=20, use_best_model=True

In [ ]:
# 9. CatBoost 直接预测 (M3a-M3d)
print("=" * 60)
print("9. CatBoost 直接预测 — M3a-M3d")
print("=" * 60)

t0 = time.perf_counter()
catboost_direct_results = {}
direct_experiments = [
    ("M3a_catboost_direct_history_raw", "history_raw"),
    ("M3b_catboost_direct_history_industry", "history_industry"),
    ("M3c_catboost_direct_history_metadata", "history_metadata"),
    ("M3d_catboost_direct_history_metadata_engineered", "history_metadata_engineered"),
]

for experiment_id, feature_set in direct_experiments:
    bundle = CATBOOST_FRAMES[feature_set]
    X_train_cb = bundle["X_train"]
    X_test_cb = bundle["X_test"]
    cat_features = bundle["cat_features"]
    oof_pred = pd.DataFrame(index=train.index, columns=TARGET_COLUMNS, dtype="float64")
    test_pred = pd.DataFrame(index=test.index, columns=TARGET_COLUMNS, dtype="float64")
    best_iters = []
    exp_start = time.perf_counter()
    print(f"  [{experiment_id}] feature_set={feature_set}")

    for target in TARGET_COLUMNS:
        print(f"    Training {_display_target_name(target)}...", end=" ", flush=True)
        target_oof = pd.Series(index=train.index, dtype="float64")
        target_best = []

        for fold, (train_idx, valid_idx) in enumerate(splits):
            Xt = X_train_cb.iloc[train_idx]
            Xv = X_train_cb.iloc[valid_idx]
            yt = y[target].iloc[train_idx]
            yv = y[target].iloc[valid_idx]
            mask = yt.notna()
            if mask.sum() < 2:
                continue
            model = CatBoostRegressor(**CB_PARAMS)
            model.fit(Xt.loc[mask], yt.loc[mask], eval_set=(Xv, yv), cat_features=cat_features)
            bi = model.get_best_iteration()
            bi = int(bi + 1) if bi is not None and bi >= 0 else CB_PARAMS["iterations"]
            target_best.append(bi)
            target_oof.iloc[valid_idx] = model.predict(Xv)

        final_iter = max(100, int(np.median(target_best)))
        full_params = {**CB_PARAMS, "iterations": final_iter, "use_best_model": False}
        full_params.pop("od_type", None)
        full_params.pop("od_wait", None)
        full_model = CatBoostRegressor(**full_params)
        mask_full = y[target].notna()
        full_model.fit(X_train_cb.loc[mask_full], y[target].loc[mask_full], cat_features=cat_features)

        oof_pred[target] = target_oof
        test_pred[target] = full_model.predict(X_test_cb)
        best_iters.append(final_iter)

        fold_scores = []
        for fold, (_, valid_idx) in enumerate(splits):
            score = r2_score(y[target].iloc[valid_idx], target_oof.iloc[valid_idx])
            fold_scores.append(score)
        print(f"iters={final_iter}, OOF R2={np.mean(fold_scores):.4f}")

    catboost_direct_results[experiment_id] = {
        "feature_set": feature_set,
        "oof": oof_pred,
        "test": test_pred,
        "best_iters": best_iters,
        "runtime_seconds": time.perf_counter() - exp_start,
    }
    print(f"  {experiment_id}: done in {catboost_direct_results[experiment_id]['runtime_seconds']:.1f}s")

m3a_oof = catboost_direct_results["M3a_catboost_direct_history_raw"]["oof"]
m3b_oof = catboost_direct_results["M3b_catboost_direct_history_industry"]["oof"]
m3c_oof = catboost_direct_results["M3c_catboost_direct_history_metadata"]["oof"]
m3d_oof = catboost_direct_results["M3d_catboost_direct_history_metadata_engineered"]["oof"]
m3a_test = catboost_direct_results["M3a_catboost_direct_history_raw"]["test"]
m3b_test = catboost_direct_results["M3b_catboost_direct_history_industry"]["test"]
m3c_test = catboost_direct_results["M3c_catboost_direct_history_metadata"]["test"]
m3d_test = catboost_direct_results["M3d_catboost_direct_history_metadata_engineered"]["test"]

print(f"✓ CatBoost direct experiments 完成 ({time.perf_counter() - t0:.1f}s)")

## 10. CatBoost 残差预测 (M4)

以 B4 规则融合结果为基准，CatBoost 学习残差 (实际值 − B4 预测值)，再将残差预测加回 B4。

In [ ]:
# ═══════════════════════════════════════════════════════════════════════
# 10. 最终模型 — CatBoost 残差预测 (M4)
# ═══════════════════════════════════════════════════════════════════════

print("=" * 60)
print("10. CatBoost 残差预测 — M4 (history_metadata_engineered, B4 baseline)")
print("=" * 60)

t0 = time.perf_counter()
m4_oof = pd.DataFrame(index=train.index, columns=TARGET_COLUMNS, dtype="float64")
m4_test = pd.DataFrame(index=test.index, columns=TARGET_COLUMNS, dtype="float64")
m4_best_iters = []
_m4_bundle = CATBOOST_FRAMES["history_metadata_engineered"]
X_train_cb = _m4_bundle["X_train"]
X_test_cb = _m4_bundle["X_test"]
cat_features = _m4_bundle["cat_features"]

for target in TARGET_COLUMNS:
    print(f"  Training residual for {_display_target_name(target)}...", end=" ", flush=True)
    # Residual = actual - B4 baseline
    residual = y[target] - b4_pred[target]
    target_oof_res = pd.Series(index=train.index, dtype="float64")
    target_best = []

    for fold, (train_idx, valid_idx) in enumerate(splits):
        Xt = X_train_cb.iloc[train_idx]; Xv = X_train_cb.iloc[valid_idx]
        rt = residual.iloc[train_idx]; rv = residual.iloc[valid_idx]
        mask = rt.notna()
        if mask.sum() < 2:
            continue
        model = CatBoostRegressor(**CB_PARAMS)
        model.fit(Xt.loc[mask], rt.loc[mask], eval_set=(Xv, rv), cat_features=cat_features)
        bi = model.get_best_iteration()
        bi = int(bi + 1) if bi is not None and bi >= 0 else CB_PARAMS["iterations"]
        target_best.append(bi)
        target_oof_res.iloc[valid_idx] = model.predict(Xv)

    # Reconstruct OOF: B4 + predicted residual
    target_oof = b4_pred[target] + target_oof_res
    m4_oof[target] = target_oof

    # Full model for test
    final_iter = max(100, int(np.median(target_best)))
    full_params = {**CB_PARAMS, "iterations": final_iter, "use_best_model": False}
    full_params.pop("od_type", None); full_params.pop("od_wait", None)
    full_model = CatBoostRegressor(**full_params)
    mask_full = residual.notna()
    full_model.fit(X_train_cb.loc[mask_full], residual.loc[mask_full], cat_features=cat_features)
    m4_test[target] = b4_test[target] + full_model.predict(X_test_cb)

    m4_best_iters.append(final_iter)
    fold_scores = []
    for fold, (_, valid_idx) in enumerate(splits):
        score = r2_score(y[target].iloc[valid_idx], target_oof.iloc[valid_idx])
        fold_scores.append(score)
    print(f"iters={final_iter}, OOF R²={np.mean(fold_scores):.4f}")

m4_scores = r2_by_target(y, m4_oof)
m4_mean = mean_target_r2(m4_scores)
print(f"\n  M4 CatBoost residual: mean_oof_r2={m4_mean:.4f}")
print(f"  Best iterations: {m4_best_iters}")
print(f"✓ M4 完成 ({time.perf_counter() - t0:.1f}s)")

## 11. OOF 融合

将 B4、M3d、M4 三个模型的 OOF 预测进行逐目标加权平均。权重通过两级网格搜索确定 (粗搜 step=0.05，细搜 step=0.01)。

In [ ]:
# ═══════════════════════════════════════════════════════════════════════
# 11. OOF 融合 — B4 + M3d + M4 逐目标加权
# ═══════════════════════════════════════════════════════════════════════

print("=" * 60)
print("11. OOF 融合 — B4 + M3d + M4")
print("=" * 60)

t0 = time.perf_counter()

oof_predictions = {"B4": b4_pred, "M3d": m3d_oof, "M4": m4_oof}
test_predictions = {"B4": b4_test, "M3d": m3d_test, "M4": m4_test}
members = ["B4", "M3d", "M4"]

def blend_target_fn(predictions: Mapping, target: str, members: Sequence, weights: Sequence) -> pd.Series:
    out = None
    for m, w in zip(members, weights):
        s = predictions[m][target].astype(float) * float(w)
        out = s.copy() if out is None else out + s
    return out.rename(target)

# Per-target grid search (coarse 0.05, then fine 0.01 around best)
blend_config = {}
coarse_weights = simplex_weights(len(members), step=0.05)
fine_weights = simplex_weights(len(members), step=0.01)

for target in TARGET_COLUMNS:
    best_score = -math.inf
    best_w = None
    # Coarse
    for w in coarse_weights:
        pred = blend_target_fn(oof_predictions, target, members, w)
        score = masked_r2(y[target], pred)
        if score > best_score:
            best_score, best_w = score, w
    # Fine around best
    for w in fine_weights:
        if any(abs(wi - ci) > 0.05 + 1e-12 for wi, ci in zip(w, best_w)):
            continue
        pred = blend_target_fn(oof_predictions, target, members, w)
        score = masked_r2(y[target], pred)
        if score > best_score:
            best_score, best_w = score, w
    blend_config[target] = {"members": list(members),
                            "weights": [float(x) for x in best_w],
                            "oof_r2": float(best_score)}
    print(f"  {_display_target_name(target)}: w={[f'{x:.2f}' for x in best_w]}, r2={best_score:.4f}")

# Apply blend
blend_oof = pd.DataFrame(index=train.index, columns=TARGET_COLUMNS, dtype="float64")
blend_test = pd.DataFrame(index=test.index, columns=TARGET_COLUMNS, dtype="float64")
for target in TARGET_COLUMNS:
    cfg = blend_config[target]
    blend_oof[target] = blend_target_fn(oof_predictions, target, cfg["members"], cfg["weights"])
    blend_test[target] = blend_target_fn(test_predictions, target, cfg["members"], cfg["weights"])

blend_scores = r2_by_target(y, blend_oof)
blend_mean = mean_target_r2(blend_scores)
print(f"\n  M6 OOF Blend: mean_oof_r2={blend_mean:.4f}")
print(f"✓ OOF 融合完成 ({time.perf_counter() - t0:.1f}s)")

## 12. 会计恒等式后处理

根据会计恒等式对预测结果进行一致性调整：
- Assets = Liabilities + Equity
- Gross Profit = Revenue − Cost of Revenue
- Operating Income = Gross Profit − Operating Expenses

在 OOF 上选择使 mean R² 最大化的调整方案。

In [ ]:
# ═══════════════════════════════════════════════════════════════════════
# 12. 会计恒等式后处理
# ═══════════════════════════════════════════════════════════════════════

print("=" * 60)
print("12. 会计恒等式后处理")
print("=" * 60)

ADJUSTMENTS = [
    "none",
    "balance_sheet_equity",       # Equity = Assets - Liabilities
    "gross_profit_identity",      # GP = Revenue - Cost
    "operating_income_identity",  # OI = GP - OpEx
    "income_statement_identity",  # GP + OI
    "balance_and_income_identity", # all three
]

def apply_accounting_adjustment(pred: pd.DataFrame, adj: str) -> pd.DataFrame:
    adjusted = pred.copy()
    if adj in {"balance_sheet_equity", "balance_and_income_identity"}:
        adjusted["Q0_TOTAL_STOCKHOLDERS_EQUITY"] = (
            adjusted["Q0_TOTAL_ASSETS"] - adjusted["Q0_TOTAL_LIABILITIES"]
        )
    if adj in {"gross_profit_identity", "income_statement_identity", "balance_and_income_identity"}:
        adjusted["Q0_GROSS_PROFIT"] = (
            adjusted["Q0_REVENUES"] - adjusted["Q0_COST_OF_REVENUES"]
        )
    if adj in {"operating_income_identity", "income_statement_identity", "balance_and_income_identity"}:
        adjusted["Q0_OPERATING_INCOME"] = (
            adjusted["Q0_GROSS_PROFIT"] - adjusted["Q0_OPERATING_EXPENSES"]
        )
    return adjusted

best_adj = "none"
best_adj_mean = -math.inf
for adj in ADJUSTMENTS:
    adj_pred = apply_accounting_adjustment(blend_oof, adj)
    scores = r2_by_target(y, adj_pred)
    m = mean_target_r2(scores)
    marker = " ✓" if m > best_adj_mean else ""
    print(f"  {adj:35s}: mean_r2={m:.4f}{marker}")
    if m > best_adj_mean:
        best_adj_mean = m
        best_adj = adj

print(f"\n  最优后处理: {best_adj} (mean_r2={best_adj_mean:.4f})")

final_oof = apply_accounting_adjustment(blend_oof, best_adj)
final_test = apply_accounting_adjustment(blend_test, best_adj)
final_scores = r2_by_target(y, final_oof)
final_mean = mean_target_r2(final_scores)

print(f"  最终 OOF mean_r2 = {final_mean:.4f}")
print(f"✓ 会计后处理完成")

## 13. 模型结果比较

In [ ]:
# ═══════════════════════════════════════════════════════════════════════
# 13. 模型结果比较
# ═══════════════════════════════════════════════════════════════════════

print("=" * 60)
print("13. 模型结果比较")
print("=" * 60)

# Collect all scores with script-compatible experiment IDs.
def _summary_from_oof(
    experiment_id: str,
    model_name: str,
    feature_set: str,
    target_strategy: str,
    pred: pd.DataFrame,
    runtime_seconds: float = 0.0,
    notes: str = "",
) -> dict[str, object]:
    scores = r2_by_target(y, pred)
    fold_means = []
    for fold, (_, valid_idx) in enumerate(splits):
        fold_scores = r2_by_target(y.iloc[valid_idx], pred.iloc[valid_idx])
        fold_means.append(mean_target_r2(fold_scores))
    row = {
        "experiment_id": experiment_id,
        "model_name": model_name,
        "feature_set": feature_set,
        "target_strategy": target_strategy,
        "mean_r2": mean_target_r2(scores),
        "fold_mean_r2_std": float(np.std(fold_means, ddof=0)),
        "runtime_seconds": float(runtime_seconds),
        "notes": notes,
    }
    row.update({f"r2_{target}": score for target, score in scores.items()})
    return row

all_rows = []
baseline_oof_map = {**base_preds, "B4": b4_pred}
for rule in ["B0", "B1", "B2", "B3", "B4"]:
    all_rows.append(
        _summary_from_oof(
            rule,
            f"{rule}_baseline",
            "row_history_only",
            "direct_rule",
            baseline_oof_map[rule],
            notes="GroupKFold OOF baseline",
        )
    )

all_rows.append(
    _summary_from_oof(
        "M1_ridge_history_raw",
        "Ridge",
        "history_raw",
        "direct",
        ridge_oof,
        t_ridge,
        "GroupKFold OOF sklearn baseline",
    )
)
all_rows.append(
    _summary_from_oof(
        "M2_hgb_history_raw",
        "HistGradientBoostingRegressor",
        "history_raw",
        "direct",
        hgb_oof,
        notes="GroupKFold OOF sklearn baseline",
    )
)

for experiment_id, feature_set in direct_experiments:
    result = catboost_direct_results[experiment_id]
    all_rows.append(
        _summary_from_oof(
            experiment_id,
            "CatBoostRegressor",
            feature_set,
            "direct",
            result["oof"],
            result["runtime_seconds"],
            f"CatBoost direct on {feature_set}; median_best_iter={int(np.median(result['best_iters']))}",
        )
    )

all_rows.append(
    _summary_from_oof(
        "M4_catboost_residual_history_metadata_engineered",
        "CatBoostRegressor",
        "history_metadata_engineered",
        "residual",
        m4_oof,
        notes=f"CatBoost residual on history_metadata_engineered; median_best_iter={int(np.median(m4_best_iters))}",
    )
)
all_rows.append(
    _summary_from_oof(
        "M6_oof_blend",
        "OOFWeightedBlend",
        "saved_oof_predictions",
        "per_target_oof_blend",
        blend_oof,
        notes="Per-target OOF blend of B4, M3d, and M4.",
    )
)
all_rows.append(
    _summary_from_oof(
        "M7_accounting_postprocess",
        "AccountingPostprocess",
        "saved_oof_predictions",
        best_adj,
        final_oof,
        notes=f"OOF-selected accounting adjustment: {best_adj}.",
    )
)

all_scores_df = pd.DataFrame(all_rows)
all_scores_df = all_scores_df.sort_values("mean_r2", ascending=False).reset_index(drop=True)

print("\n模型 OOF 得分汇总:")
print(all_scores_df[["experiment_id", "mean_r2"]].to_string(index=False))

# ── Bar chart — model comparison ───────────────────────────────────────
fig, ax = plt.subplots(figsize=(10, 5.5))
ordered = all_scores_df.sort_values("mean_r2", ascending=True)
colors = ["#BAB0AC" if v < ordered["mean_r2"].max() else "#4C78A8" for v in ordered["mean_r2"]]
labels = [_display_experiment_name(e) for e in ordered["experiment_id"]]
ax.barh(labels, ordered["mean_r2"], color=colors)
ax.set_title("模型 OOF 平均 R² 对比"); ax.set_xlabel("OOF 平均 R²")
for i, v in enumerate(ordered["mean_r2"]):
    ax.text(v, i, f" {v:.3f}", va="center", fontsize=8)
_save_fig(fig, "fig09_model_comparison.png")

# ── Heatmap — per-target scores ────────────────────────────────────────
target_cols = [c for c in all_scores_df.columns if c.startswith("r2_Q0_")]
matrix = all_scores_df.set_index("experiment_id")[target_cols]
matrix.columns = [_display_target_name(c.removeprefix("r2_")) for c in matrix.columns]
matrix.index = [_display_experiment_name(n) for n in matrix.index]
fig, ax = plt.subplots(figsize=(13, max(4.5, 0.45 * len(matrix))))
im = ax.imshow(matrix.values, aspect="auto", cmap="RdYlGn", vmin=-0.25, vmax=1.0)
ax.set_xticks(range(len(matrix.columns)))
ax.set_xticklabels(matrix.columns, rotation=35, ha="right")
ax.set_yticks(range(len(matrix.index)))
ax.set_yticklabels(matrix.index)
ax.set_title("各目标 OOF R² 热力图")
for i in range(matrix.shape[0]):
    for j in range(matrix.shape[1]):
        ax.text(j, i, f"{matrix.iloc[i, j]:.2f}", ha="center", va="center", fontsize=7)
fig.colorbar(im, ax=ax, fraction=0.02, pad=0.02)
_save_fig(fig, "fig10_target_score_heatmap.png")

# ── OOF scatter (final model) ──────────────────────────────────────────
ncols, nrows = 3, int(np.ceil(len(TARGET_COLUMNS) / 3))
fig, axes = plt.subplots(nrows, ncols, figsize=(14, 11))
axes = np.asarray(axes).reshape(-1)
for ax, target in zip(axes, TARGET_COLUMNS):
    actual = y[target].astype(float)
    pred = final_oof[target].astype(float)
    al = signed_log1p(actual); pl = signed_log1p(pred)
    ax.scatter(al, pl, s=9, alpha=0.45, color="#4C78A8", edgecolors="none")
    lo, hi = np.nanmin([al.min(), pl.min()]), np.nanmax([al.max(), pl.max()])
    ax.plot([lo, hi], [lo, hi], color="#E45756", linewidth=1)
    ax.set_title(_display_target_name(target))
    ax.set_xlabel("真实值"); ax.set_ylabel("预测值")
for ax in axes[len(TARGET_COLUMNS):]:
    ax.axis("off")
fig.suptitle("OOF 真实值与预测值散点图 (最终模型)", y=1.01, fontsize=13)
_save_fig(fig, "fig11_oof_scatter.png")

# ── Residual distribution ──────────────────────────────────────────────
residuals, rlabels = [], []
for target in TARGET_COLUMNS:
    res = final_oof[target].astype(float) - y[target].astype(float)
    scale = np.nanmedian(np.abs(y[target].astype(float))) or 1.0
    residuals.append((res / scale).replace([np.inf, -np.inf], np.nan).dropna())
    rlabels.append(_display_target_name(target))
fig, ax = plt.subplots(figsize=(12, 5.5))
ax.boxplot(residuals, labels=rlabels, showfliers=False, vert=False)
ax.set_title("各目标 OOF 残差分布 (最终模型)")
ax.set_xlabel("残差 / 目标中位绝对值")
_save_fig(fig, "fig12_residual_distribution.png")

blend_weight_rows = []
for target, cfg in blend_config.items():
    row = {"target": target}
    for member, weight in zip(cfg["members"], cfg["weights"]):
        row[member] = float(weight)
    blend_weight_rows.append(row)
blend_weight_matrix = pd.DataFrame(blend_weight_rows).set_index("target").reindex(TARGET_COLUMNS).fillna(0.0)
fig, ax = plt.subplots(figsize=(12, 5.2))
bottom = np.zeros(len(blend_weight_matrix))
colors = {"B4": "#4C78A8", "M3d": "#F58518", "M4": "#54A24B"}
for member in ["B4", "M3d", "M4"]:
    values = blend_weight_matrix[member].to_numpy(dtype="float64") if member in blend_weight_matrix else np.zeros(len(blend_weight_matrix))
    ax.bar(np.arange(len(blend_weight_matrix)), values, bottom=bottom, label=member, color=colors.get(member))
    bottom += values
ax.set_xticks(np.arange(len(blend_weight_matrix)))
ax.set_xticklabels([_display_target_name(t) for t in blend_weight_matrix.index], rotation=35, ha="right")
ax.set_ylim(0, 1.05)
ax.set_ylabel("融合权重")
ax.set_title("逐目标 OOF 融合权重")
ax.legend(ncol=3, frameon=True)
_save_fig(fig, "fig13_blend_weights.png")

print(f"\n✓ 模型比较图表已保存至 {FIGURES}")

## 14. 生成 submission.csv

In [ ]:
# ═══════════════════════════════════════════════════════════════════════
# 14. 生成 submission.csv
# ═══════════════════════════════════════════════════════════════════════

print("=" * 60)
print("14. 生成 submission.csv")
print("=" * 60)

# Build submission from final_test (rows match test/sample_submission ordering)
submission = sample_submission[[ID_COLUMN]].copy()
assert test[ID_COLUMN].equals(sample_submission[ID_COLUMN]), "ID order mismatch"

# Diagnostic: check for NaN before building submission
print("  最终预测 NaN 检查:")
for col in TARGET_COLUMNS:
    nan_count = int(final_test[col].isna().sum())
    inf_count = int(np.isinf(final_test[col].astype(float)).sum())
    if nan_count > 0 or inf_count > 0:
        print(f"    {col}: NaN={nan_count}, Inf={inf_count}")

for col in SUBMISSION_COLUMNS:
    if col == ID_COLUMN:
        continue
    submission[col] = final_test[col].values

# Validate
assert list(submission.columns) == SUBMISSION_COLUMNS, "column order mismatch"
assert submission.shape == sample_submission.shape
assert submission[ID_COLUMN].equals(sample_submission[ID_COLUMN])
vals = submission.drop(columns=[ID_COLUMN]).to_numpy(dtype="float64")
assert not np.isnan(vals).any(), "NaN in submission"
assert not np.isinf(vals).any(), "Inf in submission"

# Save both standalone-root and deliverables copies.
output_path = ROOT / "submission.csv"
deliverable_submission_path = DELIVERABLES / "submission.csv"
submission.to_csv(output_path, index=False)
submission.to_csv(deliverable_submission_path, index=False)
print(f"  submission.csv 已保存至: {output_path}")
print(f"  deliverables/submission.csv 已保存至: {deliverable_submission_path}")
print(f"  形状: {submission.shape}")
print(f"  列:   {list(submission.columns)}")
print(f"\n  前5行预览:")
print(submission.head().to_string(index=False))
print(f"\n✓ submission.csv 生成完成")

## 15. 导出同源结果文件

In [ ]:
# ============================================================================
# 15. Export audit/model artifacts for the report and delivery checks
# ============================================================================
print("=" * 60)
print("15. 导出同源结果文件")
print("=" * 60)

import importlib.metadata as importlib_metadata
import platform
import subprocess
import sys
from datetime import datetime, timezone

EXPORT_TS = datetime.now(timezone.utc).isoformat()

def _clean_for_json(obj):
    if isinstance(obj, dict):
        return {str(k): _clean_for_json(v) for k, v in obj.items()}
    if isinstance(obj, (list, tuple)):
        return [_clean_for_json(v) for v in obj]
    if isinstance(obj, pd.DataFrame):
        return _clean_for_json(obj.to_dict(orient="records"))
    if isinstance(obj, pd.Series):
        return _clean_for_json(obj.to_dict())
    if isinstance(obj, np.ndarray):
        return _clean_for_json(obj.tolist())
    if isinstance(obj, (np.integer,)):
        return int(obj)
    if isinstance(obj, (np.floating, float)):
        val = float(obj)
        return val if math.isfinite(val) else None
    if isinstance(obj, (np.bool_, bool)):
        return bool(obj)
    try:
        if pd.isna(obj):
            return None
    except Exception:
        pass
    return obj

def write_json(path: Path, payload) -> None:
    path.parent.mkdir(parents=True, exist_ok=True)
    path.write_text(
        json.dumps(_clean_for_json(payload), ensure_ascii=False, indent=2, allow_nan=False),
        encoding="utf-8",
    )

def write_csv(path: Path, frame: pd.DataFrame) -> None:
    path.parent.mkdir(parents=True, exist_ok=True)
    frame.to_csv(path, index=False)

def sha256_file(path: Path) -> str:
    h = hashlib.sha256()
    with path.open("rb") as f:
        for chunk in iter(lambda: f.read(1024 * 1024), b""):
            h.update(chunk)
    return h.hexdigest()

def file_metadata(path: Path) -> dict[str, object]:
    stat = path.stat()
    return {
        "path": str(path),
        "name": path.name,
        "size_bytes": int(stat.st_size),
        "mtime_utc": pd.to_datetime(stat.st_mtime, unit="s", utc=True).isoformat(),
        "sha256": sha256_file(path),
        "exists": True,
    }

def _git_commit() -> str | None:
    try:
        result = subprocess.run(
            ["git", "rev-parse", "--short", "HEAD"],
            cwd=ROOT,
            check=True,
            capture_output=True,
            text=True,
        )
        return result.stdout.strip()
    except Exception:
        return None

def _package_versions(names: Sequence[str]) -> tuple[dict[str, str], list[str]]:
    installed, missing = {}, []
    for name in names:
        try:
            installed[name] = importlib_metadata.version(name)
        except importlib_metadata.PackageNotFoundError:
            missing.append(name)
    return installed, missing

def _read_optional_text(path: Path, max_chars: int = 5000) -> dict[str, object]:
    if not path.exists():
        return {"exists": False, "path": str(path)}
    text = path.read_text(encoding="utf-8", errors="replace")
    return {
        "exists": True,
        "char_count": len(text),
        "line_count": text.count("\n") + (1 if text else 0),
        "preview": text[:max_chars],
    }

def _feature_columns(frame: pd.DataFrame) -> list[str]:
    return [c for c in frame.columns if c not in {ID_COLUMN, *TARGET_COLUMNS}]

def _common_feature_columns(train_frame: pd.DataFrame, test_frame: pd.DataFrame) -> list[str]:
    test_cols = set(test_frame.columns)
    return [c for c in train_frame.columns if c in test_cols and c not in {ID_COLUMN, *TARGET_COLUMNS}]

def _quarter_of(column: str) -> str | None:
    m = QUARTER_VALUE_PATTERN.match(column) or QUARTER_FYE_PATTERN.match(column)
    return f"Q{m.group(1)}" if m else None

def _schema_summary(train_frame: pd.DataFrame, test_frame: pd.DataFrame, sample: pd.DataFrame) -> pd.DataFrame:
    rows = []
    for name, frame, target_count in [
        ("train", train_frame, len(TARGET_COLUMNS)),
        ("test", test_frame, 0),
        ("sample_submission", sample, 0),
    ]:
        rows.append({
            "dataset": name,
            "rows": int(frame.shape[0]),
            "cols": int(frame.shape[1]),
            "id_unique": bool(frame[ID_COLUMN].is_unique) if ID_COLUMN in frame.columns else False,
            "missing_cells": int(frame.isna().sum().sum()),
            "missing_rate": float(frame.isna().sum().sum() / frame.size),
            "numeric_cols": int(frame.select_dtypes(include=[np.number]).shape[1]),
            "object_cols": int(frame.select_dtypes(include=["object"]).shape[1]),
            "feature_cols": len(_feature_columns(frame)) if name != "sample_submission" else 0,
            "target_cols": target_count,
        })
    return pd.DataFrame(rows)

def _dtype_summary(train_frame: pd.DataFrame, test_frame: pd.DataFrame) -> pd.DataFrame:
    rows = []
    for col in sorted(set(train_frame.columns) | set(test_frame.columns)):
        tr = train_frame[col] if col in train_frame.columns else pd.Series(dtype="float64")
        te = test_frame[col] if col in test_frame.columns else pd.Series(dtype="float64")
        rows.append({
            "column": col,
            "train_dtype": str(tr.dtype),
            "test_dtype": str(te.dtype),
            "train_non_null": int(tr.notna().sum()) if col in train_frame.columns else 0,
            "test_non_null": int(te.notna().sum()) if col in test_frame.columns else 0,
            "train_unique": int(tr.nunique(dropna=True)) if col in train_frame.columns else 0,
            "test_unique": int(te.nunique(dropna=True)) if col in test_frame.columns else 0,
            "train_missing_rate": float(tr.isna().mean()) if col in train_frame.columns else np.nan,
            "test_missing_rate": float(te.isna().mean()) if col in test_frame.columns else np.nan,
        })
    return pd.DataFrame(rows)

def _missing_rate_by_column(train_frame: pd.DataFrame, test_frame: pd.DataFrame) -> pd.DataFrame:
    rows = []
    for col in sorted(set(train_frame.columns) | set(test_frame.columns)):
        parts = []
        if col in train_frame.columns:
            parts.append(train_frame[col])
        if col in test_frame.columns:
            parts.append(test_frame[col])
        if col == ID_COLUMN:
            group = "id"
        elif col in TARGET_COLUMNS:
            group = "target"
        elif col in METADATA_COLUMNS:
            group = "metadata"
        elif _quarter_of(col):
            group = "historical"
        else:
            group = "other"
        rows.append({
            "column": col,
            "group": group,
            "in_train": bool(col in train_frame.columns),
            "in_test": bool(col in test_frame.columns),
            "train_missing_rate": float(train_frame[col].isna().mean()) if col in train_frame.columns else np.nan,
            "test_missing_rate": float(test_frame[col].isna().mean()) if col in test_frame.columns else np.nan,
            "combined_missing_rate": float(pd.concat(parts, axis=0).isna().mean()) if parts else np.nan,
        })
    return pd.DataFrame(rows)

def _missing_rate_by_quarter(train_frame: pd.DataFrame, test_frame: pd.DataFrame) -> pd.DataFrame:
    common = _common_feature_columns(train_frame, test_frame)
    rows = []
    for q in HISTORICAL_QUARTERS:
        q_cols = [c for c in common if c.startswith(f"{q}_")]
        if not q_cols:
            continue
        rates = pd.Series({c: pd.concat([train_frame[c], test_frame[c]], axis=0).isna().mean() for c in q_cols})
        rows.append({
            "quarter": q,
            "avg_missing_rate": float(rates.mean()),
            "median_missing_rate": float(rates.median()),
            "max_missing_rate": float(rates.max()),
            "min_missing_rate": float(rates.min()),
            "observed_columns": int(len(q_cols)),
        })
    return pd.DataFrame(rows)

def _category_summary(train_frame: pd.DataFrame, test_frame: pd.DataFrame) -> pd.DataFrame:
    rows = []
    cols = sorted(set(train_frame.select_dtypes(include=["object"]).columns) | set(test_frame.select_dtypes(include=["object"]).columns))
    for col in cols:
        tr = train_frame[col] if col in train_frame.columns else pd.Series(dtype="object")
        te = test_frame[col] if col in test_frame.columns else pd.Series(dtype="object")
        trv = tr.dropna().astype(str)
        tev = te.dropna().astype(str)
        tr_set, te_set = set(trv.unique()), set(tev.unique())
        rows.append({
            "column": col,
            "train_unique": int(len(tr_set)),
            "test_unique": int(len(te_set)),
            "shared_categories": int(len(tr_set & te_set)),
            "train_only_categories": int(len(tr_set - te_set)),
            "test_only_categories": int(len(te_set - tr_set)),
            "train_missing_rate": float(tr.isna().mean()) if col in train_frame.columns else np.nan,
            "test_missing_rate": float(te.isna().mean()) if col in test_frame.columns else np.nan,
            "train_top_value": trv.value_counts().index[0] if len(trv) else None,
            "train_top_freq": int(trv.value_counts().iloc[0]) if len(trv) else 0,
            "test_top_value": tev.value_counts().index[0] if len(tev) else None,
            "test_top_freq": int(tev.value_counts().iloc[0]) if len(tev) else 0,
        })
    return pd.DataFrame(rows)

def _duplicate_summary(train_frame: pd.DataFrame, test_frame: pd.DataFrame) -> pd.DataFrame:
    common = _common_feature_columns(train_frame, test_frame)
    rows = []
    scopes = [
        ("train_non_id", train_frame.drop(columns=[ID_COLUMN])),
        ("train_common_features", train_frame[common]),
        ("test_common_features", test_frame[common]),
    ]
    for scope, frame in scopes:
        hashes = stable_row_hash(replace_inf_with_nan(frame.copy()))
        counts = hashes.value_counts()
        dup_groups = counts[counts > 1]
        rows.append({
            "scope": scope,
            "duplicate_rows": int((dup_groups - 1).sum()) if not dup_groups.empty else 0,
            "duplicate_groups": int(dup_groups.shape[0]),
            "max_group_size": int(dup_groups.max()) if not dup_groups.empty else 1,
            "notes": "exact duplicate rows within the same dataset scope",
        })
    train_hash = stable_row_hash(replace_inf_with_nan(train_frame[common].copy()))
    test_hash = stable_row_hash(replace_inf_with_nan(test_frame[common].copy()))
    shared = set(train_hash.unique()) & set(test_hash.unique())
    train_matches = int(train_hash.isin(shared).sum())
    test_matches = int(test_hash.isin(shared).sum())
    rows.extend([
        {
            "scope": "train_rows_matching_test_common_features",
            "duplicate_rows": train_matches,
            "duplicate_groups": int(len(shared)),
            "max_group_size": int(train_hash[train_hash.isin(shared)].value_counts().max()) if train_matches else 1,
            "notes": "train rows whose common feature hash also appears in test",
        },
        {
            "scope": "test_rows_matching_train_common_features",
            "duplicate_rows": test_matches,
            "duplicate_groups": int(len(shared)),
            "max_group_size": int(test_hash[test_hash.isin(shared)].value_counts().max()) if test_matches else 1,
            "notes": "test rows whose common feature hash also appears in train",
        },
    ])
    return pd.DataFrame(rows)

def _target_summary(train_frame: pd.DataFrame) -> pd.DataFrame:
    rows = []
    for target in TARGET_COLUMNS:
        s = pd.to_numeric(train_frame[target], errors="coerce").replace([np.inf, -np.inf], np.nan)
        rows.append({
            "target": target,
            "count": int(s.notna().sum()),
            "missing": int(s.isna().sum()),
            "mean": float(s.mean()),
            "std": float(s.std()),
            "min": float(s.min()),
            "p01": float(s.quantile(0.01)),
            "p05": float(s.quantile(0.05)),
            "median": float(s.median()),
            "p95": float(s.quantile(0.95)),
            "p99": float(s.quantile(0.99)),
            "max": float(s.max()),
            "skew": float(s.skew()),
            "kurtosis": float(s.kurtosis()),
            "positive_rate": float((s.dropna() > 0).mean()),
        })
    return pd.DataFrame(rows)

def _numeric_extreme_summary(train_frame: pd.DataFrame, test_frame: pd.DataFrame) -> pd.DataFrame:
    rows = []
    for dataset, frame in [("train", train_frame), ("test", test_frame)]:
        for col in frame.select_dtypes(include=[np.number]).columns:
            s = pd.to_numeric(frame[col], errors="coerce")
            arr = s.to_numpy(dtype="float64")
            finite = s[np.isfinite(arr)]
            rows.append({
                "dataset": dataset,
                "column": col,
                "count": int(s.notna().sum()),
                "missing": int(s.isna().sum()),
                "pos_inf_count": int(np.isposinf(arr).sum()),
                "neg_inf_count": int(np.isneginf(arr).sum()),
                "negative_count": int((finite < 0).sum()),
                "negative_rate": float((finite < 0).mean()) if len(finite) else np.nan,
                "zero_count": int((finite == 0).sum()),
                "min": float(finite.min()) if len(finite) else np.nan,
                "p01": float(finite.quantile(0.01)) if len(finite) else np.nan,
                "p05": float(finite.quantile(0.05)) if len(finite) else np.nan,
                "median": float(finite.median()) if len(finite) else np.nan,
                "p95": float(finite.quantile(0.95)) if len(finite) else np.nan,
                "p99": float(finite.quantile(0.99)) if len(finite) else np.nan,
                "max": float(finite.max()) if len(finite) else np.nan,
            })
    return pd.DataFrame(rows)

def _metadata_target_correlation(train_frame: pd.DataFrame) -> pd.DataFrame:
    rows = []
    numeric_metadata = [c for c in METADATA_COLUMNS if c in train_frame.columns and pd.api.types.is_numeric_dtype(train_frame[c])]
    for feature in numeric_metadata:
        for target in TARGET_COLUMNS:
            pair = train_frame[[feature, target]].replace([np.inf, -np.inf], np.nan).dropna()
            corr = np.nan
            if len(pair) >= 3 and pair[feature].nunique() > 1 and pair[target].nunique() > 1:
                corr = float(pair[feature].corr(pair[target]))
            rows.append({
                "metadata_column": feature,
                "target": target,
                "n_obs": int(len(pair)),
                "pearson_corr": corr,
                "abs_corr": abs(corr) if pd.notna(corr) else np.nan,
            })
    return pd.DataFrame(rows)

def _accounting_identity_summary(train_frame: pd.DataFrame, test_frame: pd.DataFrame) -> tuple[pd.DataFrame, pd.Series]:
    common = _common_feature_columns(train_frame, test_frame)
    rows, errors = [], []
    for q in HISTORICAL_QUARTERS:
        assets_col = f"{q}_TOTAL_ASSETS"
        liabilities_col = f"{q}_TOTAL_LIABILITIES"
        equity_col = f"{q}_TOTAL_STOCKHOLDERS_EQUITY"
        cols = [assets_col, liabilities_col, equity_col]
        if not all(c in common for c in cols):
            continue
        combined = pd.concat([train_frame[cols], test_frame[cols]], axis=0).replace([np.inf, -np.inf], np.nan)
        rel = (combined[assets_col] - combined[liabilities_col] - combined[equity_col]) / combined[assets_col].abs().replace(0, np.nan)
        errors.append(rel.rename(q))
        rows.append({
            "quarter": q,
            "n_obs": int(rel.notna().sum()),
            "mean_abs_relative_error": float(rel.abs().mean()),
            "median_abs_relative_error": float(rel.abs().median()),
            "p95_abs_relative_error": float(rel.abs().quantile(0.95)),
            "within_1e-6": float((rel.abs() <= 1e-6).mean()),
            "within_1e-4": float((rel.abs() <= 1e-4).mean()),
            "within_1e-2": float((rel.abs() <= 1e-2).mean()),
        })
    return pd.DataFrame(rows), pd.concat(errors, axis=0) if errors else pd.Series(dtype="float64")

def _lag_correlation(train_frame: pd.DataFrame) -> pd.DataFrame:
    rows = []
    for q in HISTORICAL_QUARTERS:
        row = {"quarter": q}
        for target in TARGET_COLUMNS:
            metric = target.removeprefix("Q0_")
            col = f"{q}_{metric}"
            if col in train_frame.columns:
                pair = train_frame[[col, target]].replace([np.inf, -np.inf], np.nan).dropna()
                row[target] = float(pair[col].corr(pair[target])) if len(pair) >= 3 and pair[col].nunique() > 1 else np.nan
            else:
                row[target] = np.nan
        rows.append(row)
    return pd.DataFrame(rows)

def _score_summary_row(experiment_id: str, model_name: str, feature_set: str, target_strategy: str,
                       pred: pd.DataFrame, runtime_seconds: float = 0.0, notes: str = "") -> dict[str, object]:
    scores = r2_by_target(y, pred)
    fold_means = []
    for _, valid_idx in splits:
        fold_scores = r2_by_target(y.iloc[valid_idx], pred.iloc[valid_idx])
        fold_means.append(mean_target_r2(fold_scores))
    row = {
        "experiment_id": experiment_id,
        "model_name": model_name,
        "feature_set": feature_set,
        "target_strategy": target_strategy,
        "mean_r2": mean_target_r2(scores),
        "fold_mean_r2_std": float(np.std(fold_means, ddof=0)),
        "runtime_seconds": float(runtime_seconds),
        "notes": notes,
    }
    row.update({f"r2_{target}": score for target, score in scores.items()})
    return row

def _fold_score_rows(experiment_id: str, model_name: str, feature_set: str, target_strategy: str,
                     pred: pd.DataFrame, runtime_seconds: float = 0.0, notes: str = "") -> pd.DataFrame:
    rows, fold_means = [], []
    full_scores = r2_by_target(y, pred)
    mean_r2 = mean_target_r2(full_scores)
    for fold, (_, valid_idx) in enumerate(splits):
        fold_scores = r2_by_target(y.iloc[valid_idx], pred.iloc[valid_idx])
        fold_means.append(mean_target_r2(fold_scores))
        for target, score in fold_scores.items():
            rows.append({
                "experiment_id": experiment_id,
                "timestamp": EXPORT_TS,
                "model_name": model_name,
                "feature_set": feature_set,
                "target_strategy": target_strategy,
                "fold": int(fold),
                "target": target,
                "r2": float(score),
                "mean_r2": float(mean_r2),
                "std_r2": np.nan,
                "runtime_seconds": float(runtime_seconds),
                "seed": 42,
                "git_commit": GIT_COMMIT,
                "notes": notes,
            })
    out = pd.DataFrame(rows)
    out["std_r2"] = float(np.std(fold_means, ddof=0))
    return out

def _write_oof(path: Path, pred: pd.DataFrame) -> None:
    out = pd.DataFrame({ID_COLUMN: train[ID_COLUMN].values, "group_id": groups.values, "fold": fold_assignments.astype(int).values})
    for target in TARGET_COLUMNS:
        out[f"actual_{target}"] = y[target].values
        out[f"pred_{target}"] = pred[target].values
    write_csv(path, out)

def _write_prediction(path: Path, pred: pd.DataFrame) -> None:
    out = pd.DataFrame({ID_COLUMN: test[ID_COLUMN].values})
    for target in TARGET_COLUMNS:
        out[target] = pred[target].values
    write_csv(path, out)

def _submission_from_predictions(pred: pd.DataFrame) -> pd.DataFrame:
    out = sample_submission[[ID_COLUMN]].copy()
    for col in SUBMISSION_COLUMNS:
        if col == ID_COLUMN:
            continue
        out[col] = pred[col].values
    return out

GIT_COMMIT = _git_commit()

raw_manifest = {
    "timestamp_utc": EXPORT_TS,
    "project_root": str(ROOT),
    "raw_files": {},
    "csv_summary": {},
    "text_summary": {},
}
for name in ["train.csv", "test.csv", "sample_submission.csv"]:
    path = ROOT / name
    raw_manifest["raw_files"][name] = file_metadata(path)
    frame = pd.read_csv(path)
    raw_manifest["csv_summary"][name] = {"rows": int(frame.shape[0]), "cols": int(frame.shape[1]), "columns": list(frame.columns)}
for name in ["data_dictionary.txt"]:
    path = ROOT / name
    if path.exists():
        raw_manifest["raw_files"][name] = file_metadata(path)
    raw_manifest["text_summary"][name] = _read_optional_text(path)
write_json(RESULTS / "input_manifest.json", raw_manifest)

required_packages = [
    "pandas", "numpy", "scipy", "matplotlib", "scikit-learn", "catboost",
    "nbconvert", "nbformat", "ipykernel", "python-docx", "openpyxl", "pytest",
]
installed_versions, missing_packages = _package_versions(required_packages)
write_json(RESULTS / "environment_audit.json", {
    "timestamp_utc": EXPORT_TS,
    "conda_environment_name": os.environ.get("CONDA_DEFAULT_ENV"),
    "python_executable": sys.executable,
    "python_version": sys.version.split()[0],
    "platform": platform.platform(),
    "required_packages": required_packages,
    "installed_versions": installed_versions,
    "missing_packages": missing_packages,
})

raw_train = pd.read_csv(ROOT / "train.csv")
raw_test = pd.read_csv(ROOT / "test.csv")
schema_summary = _schema_summary(raw_train, raw_test, sample_submission)
dtype_summary = _dtype_summary(raw_train, raw_test)
missing_by_column = _missing_rate_by_column(raw_train, raw_test)
missing_by_quarter = _missing_rate_by_quarter(raw_train, raw_test)
category_summary = _category_summary(raw_train, raw_test)
duplicate_summary = _duplicate_summary(raw_train, raw_test)
target_summary = _target_summary(raw_train)
numeric_extreme_summary = _numeric_extreme_summary(raw_train, raw_test)
metadata_target_correlation = _metadata_target_correlation(raw_train)
accounting_identity_summary, accounting_errors = _accounting_identity_summary(raw_train, raw_test)
lag_correlation = _lag_correlation(raw_train)
target_correlation = raw_train[TARGET_COLUMNS].replace([np.inf, -np.inf], np.nan).corr()

write_csv(TABLES / "schema_summary.csv", schema_summary)
write_csv(TABLES / "missing_rate_by_column.csv", missing_by_column)
write_csv(TABLES / "missing_rate_by_quarter.csv", missing_by_quarter)
write_csv(TABLES / "dtype_summary.csv", dtype_summary)
write_csv(TABLES / "category_summary.csv", category_summary)
write_csv(TABLES / "duplicate_summary.csv", duplicate_summary)
write_csv(TABLES / "target_summary.csv", target_summary)
write_csv(TABLES / "numeric_extreme_summary.csv", numeric_extreme_summary)
write_csv(TABLES / "metadata_target_correlation.csv", metadata_target_correlation)
write_csv(TABLES / "accounting_identity_summary.csv", accounting_identity_summary)
write_csv(TABLES / "lag_correlation.csv", lag_correlation)
write_csv(TABLES / "target_correlation.csv", target_correlation.reset_index().rename(columns={"index": "target"}))

write_json(RESULTS / "data_audit.json", {
    "timestamp_utc": EXPORT_TS,
    "train_shape": list(raw_train.shape),
    "test_shape": list(raw_test.shape),
    "sample_submission_shape": list(sample_submission.shape),
    "id_unique_train": bool(raw_train[ID_COLUMN].is_unique),
    "id_unique_test": bool(raw_test[ID_COLUMN].is_unique),
    "id_unique_sample_submission": bool(sample_submission[ID_COLUMN].is_unique),
    "target_columns": TARGET_COLUMNS,
    "submission_column_order_matches": list(sample_submission.columns) == SUBMISSION_COLUMNS,
    "missing": {
        "train_missing_cells": int(raw_train.isna().sum().sum()),
        "test_missing_cells": int(raw_test.isna().sum().sum()),
        "train_missing_rate": float(raw_train.isna().sum().sum() / raw_train.size),
        "test_missing_rate": float(raw_test.isna().sum().sum() / raw_test.size),
        "top_missing_columns": missing_by_column.sort_values("combined_missing_rate", ascending=False).head(20)[["column", "combined_missing_rate"]].to_dict(orient="records"),
        "quarter_missing_rate": missing_by_quarter.to_dict(orient="records"),
    },
    "duplicates": duplicate_summary.to_dict(orient="records"),
    "inf_values": numeric_extreme_summary.groupby("dataset")[["pos_inf_count", "neg_inf_count"]].sum().astype(int).to_dict(orient="index"),
    "constant_columns_train": [c for c in raw_train.columns if raw_train[c].nunique(dropna=True) <= 1],
    "constant_columns_test": [c for c in raw_test.columns if raw_test[c].nunique(dropna=True) <= 1],
    "near_constant_columns_train": [c for c in raw_train.columns if len(raw_train[c].dropna()) and raw_train[c].dropna().value_counts(normalize=True).iloc[0] >= 0.99],
    "near_constant_columns_test": [c for c in raw_test.columns if len(raw_test[c].dropna()) and raw_test[c].dropna().value_counts(normalize=True).iloc[0] >= 0.99],
    "accounting_identity": accounting_identity_summary.to_dict(orient="records"),
    "accounting_identity_overall": {
        "n_obs": int(accounting_errors.notna().sum()),
        "mean_abs_relative_error": float(accounting_errors.abs().mean()) if len(accounting_errors) else np.nan,
        "median_abs_relative_error": float(accounting_errors.abs().median()) if len(accounting_errors) else np.nan,
        "p95_abs_relative_error": float(accounting_errors.abs().quantile(0.95)) if len(accounting_errors) else np.nan,
    },
})

baseline_oof_map = {**base_preds, "B4": b4_pred}
score_specs = [
    ("B0", "B0_baseline", "row_history_only", "direct_rule", baseline_oof_map["B0"], t_baseline, "Mean target by train fold."),
    ("B1", "B1_baseline", "row_history_only", "direct_rule", baseline_oof_map["B1"], t_baseline, "Recent quarter copy baseline."),
    ("B2", "B2_baseline", "row_history_only", "direct_rule", baseline_oof_map["B2"], t_baseline, "Seasonal quarter copy baseline."),
    ("B3", "B3_baseline", "row_history_only", "direct_rule", baseline_oof_map["B3"], t_baseline, "Short trend extrapolation baseline."),
    ("B4", "B4_baseline_blend", "row_history_only", "baseline_oof_blend", b4_pred, t_baseline, "Per-target OOF blend of B1/B2/B3."),
    ("M1_ridge_history_raw", "Ridge", "history_raw", "direct", ridge_oof, t_ridge, "Ridge with median imputation, robust scaling and one-hot categorical encoding."),
    ("M2_hgb_history_raw", "HistGradientBoostingRegressor", "history_raw", "direct", hgb_oof, 0.0, "Sklearn nonlinear tree baseline on raw history features."),
]
for experiment_id, feature_set in direct_experiments:
    result = catboost_direct_results[experiment_id]
    score_specs.append((
        experiment_id,
        "CatBoostRegressor",
        feature_set,
        "direct",
        result["oof"],
        result["runtime_seconds"],
        f"CatBoost direct; median_best_iter={int(np.median(result['best_iters']))}.",
    ))
score_specs.extend([
    ("M4_catboost_residual_history_metadata_engineered", "CatBoostRegressor", "history_metadata_engineered", "residual", m4_oof, 0.0, f"CatBoost residual; median_best_iter={int(np.median(m4_best_iters))}."),
    ("M6_oof_blend", "OOFWeightedBlend", "saved_oof_predictions", "per_target_oof_blend", blend_oof, 0.0, "Per-target OOF blend of B4, M3d and M4."),
    ("M7_accounting_postprocess", "AccountingPostprocess", "saved_oof_predictions", best_adj, final_oof, 0.0, f"OOF-selected accounting adjustment: {best_adj}."),
])

summary_rows = [_score_summary_row(*spec) for spec in score_specs]
summary_df = pd.DataFrame(summary_rows)
all_model_scores = summary_df.sort_values("mean_r2", ascending=False).reset_index(drop=True)
write_csv(TABLES / "all_model_scores.csv", all_model_scores)
write_csv(TABLES / "baseline_scores.csv", summary_df[summary_df["experiment_id"].isin(["B0", "B1", "B2", "B3", "B4"])])
write_csv(TABLES / "sklearn_model_scores.csv", summary_df[summary_df["experiment_id"].isin(["M1_ridge_history_raw", "M2_hgb_history_raw"])])
write_csv(TABLES / "catboost_model_scores.csv", summary_df[summary_df["experiment_id"].str.startswith("M3") | summary_df["experiment_id"].str.startswith("M4")])
write_csv(TABLES / "final_model_scores.csv", summary_df[summary_df["experiment_id"].isin(["M6_oof_blend", "M7_accounting_postprocess"])])
for _, row in summary_df.iterrows():
    eid = row["experiment_id"]
    if str(eid).startswith(("M1", "M2", "M3", "M4")):
        write_csv(TABLES / f"{eid}_scores.csv", pd.DataFrame([row]))

cv_frames = []
for spec in score_specs:
    cv = _fold_score_rows(*spec)
    cv_frames.append(cv)
    write_csv(CV_SCORES / f"{spec[0].lower()}.csv", cv)
write_csv(RESULTS / "experiment_log.csv", pd.concat(cv_frames, ignore_index=True))

baseline_oof = pd.DataFrame({ID_COLUMN: train[ID_COLUMN].values, "group_id": groups.values, "fold": fold_assignments.astype(int).values})
for target in TARGET_COLUMNS:
    baseline_oof[f"actual_{target}"] = y[target].values
for rule, pred in baseline_oof_map.items():
    for target in TARGET_COLUMNS:
        baseline_oof[f"{rule}_{target}"] = pred[target].values
write_csv(OOF_DIR / "baseline_oof.csv", baseline_oof)

for name, pred in {
    "m1_ridge_history_raw.csv": ridge_oof,
    "m2_hgb_history_raw.csv": hgb_oof,
    "m3a_catboost_direct_history_raw.csv": m3a_oof,
    "m3b_catboost_direct_history_industry.csv": m3b_oof,
    "m3c_catboost_direct_history_metadata.csv": m3c_oof,
    "m3d_catboost_direct_history_metadata_engineered.csv": m3d_oof,
    "m4_catboost_residual_history_metadata_engineered.csv": m4_oof,
    "m6_oof_blend.csv": blend_oof,
    "m7_accounting_postprocess.csv": final_oof,
}.items():
    _write_oof(OOF_DIR / name, pred)

for name, pred in {
    "baseline_b4_test_predictions.csv": b4_test,
    "m1_ridge_history_raw_test_predictions.csv": ridge_test,
    "m2_hgb_history_raw_test_predictions.csv": hgb_test,
    "m3a_catboost_direct_history_raw_test_predictions.csv": m3a_test,
    "m3b_catboost_direct_history_industry_test_predictions.csv": m3b_test,
    "m3c_catboost_direct_history_metadata_test_predictions.csv": m3c_test,
    "m3d_catboost_direct_history_metadata_engineered_test_predictions.csv": m3d_test,
    "m4_catboost_residual_history_metadata_engineered_test_predictions.csv": m4_test,
}.items():
    _write_prediction(PREDICTIONS_DIR / name, pred)
write_csv(PREDICTIONS_DIR / "m6_oof_blend_test_predictions.csv", _submission_from_predictions(blend_test))
write_csv(PREDICTIONS_DIR / "final_submission_candidate.csv", submission)

blend_table = pd.DataFrame([
    {
        "target": target,
        "members": ",".join(cfg["members"]),
        "weights": ",".join(f"{w:.2f}" for w in cfg["weights"]),
        "oof_r2": cfg["oof_r2"],
        "coarse_step": 0.05,
        "fine_step": 0.01,
        "fine_radius": 0.05,
    }
    for target, cfg in blend_config.items()
])
write_csv(TABLES / "blend_scores.csv", blend_table)

accounting_rows = []
for adj in ADJUSTMENTS:
    adj_pred = apply_accounting_adjustment(blend_oof, adj)
    scores = r2_by_target(y, adj_pred)
    row = {"adjustment": adj, "mean_r2": mean_target_r2(scores)}
    row.update({f"r2_{target}": score for target, score in scores.items()})
    accounting_rows.append(row)
write_csv(TABLES / "accounting_postprocess_scores.csv", pd.DataFrame(accounting_rows).sort_values("mean_r2", ascending=False).reset_index(drop=True))

write_csv(TABLES / "final_prediction_identity_summary.csv", pd.DataFrame([
    {
        "target_frame": "final_oof",
        "balance_sheet_mean_abs_error": float((final_oof["Q0_TOTAL_ASSETS"] - final_oof["Q0_TOTAL_LIABILITIES"] - final_oof["Q0_TOTAL_STOCKHOLDERS_EQUITY"]).abs().mean()),
        "gross_profit_mean_abs_error": float((final_oof["Q0_REVENUES"] - final_oof["Q0_COST_OF_REVENUES"] - final_oof["Q0_GROSS_PROFIT"]).abs().mean()),
        "operating_income_mean_abs_error": float((final_oof["Q0_GROSS_PROFIT"] - final_oof["Q0_OPERATING_EXPENSES"] - final_oof["Q0_OPERATING_INCOME"]).abs().mean()),
    }
]))

write_json(CONFIGS / "baseline_blend_weights.json", {"experiment_id": "B4", "weights": b4_config, "timestamp_utc": EXPORT_TS})
write_json(CONFIGS / "blend_weights.json", {
    "experiment_id": "M6_oof_blend",
    "members": ["B4", "M3d", "M4"],
    "selection": "per_target_oof_r2",
    "weights": blend_config,
    "timestamp_utc": EXPORT_TS,
})
write_json(CONFIGS / "accounting_postprocess.json", {
    "experiment_id": "M7_accounting_postprocess",
    "selected_adjustment": best_adj,
    "selection": "max_mean_oof_r2_among_predefined_accounting_adjustments",
    "timestamp_utc": EXPORT_TS,
})

final_experiment_id = "M7_accounting_postprocess" if best_adj != "none" else "M6_oof_blend"
best_row = summary_df.loc[summary_df["experiment_id"].eq(final_experiment_id)].iloc[0]
write_json(CONFIGS / "best_config.json", {
    "status": "generated_by_standalone_notebook",
    "best_experiment_id": final_experiment_id,
    "best_mean_oof_r2": float(best_row["mean_r2"]),
    "best_fold_mean_r2_std": float(best_row["fold_mean_r2_std"]),
    "model_family": "OOF blend with optional accounting postprocess",
    "catboost_params": CB_PARAMS,
    "timestamp_utc": EXPORT_TS,
    "notes": "All final parameters are fixed in the standalone notebook; blend and accounting choices are selected on OOF only.",
})

submission_sha = sha256_file(deliverable_submission_path)
write_json(RESULTS / "final_submission_manifest.json", {
    "submission_path": "deliverables/submission.csv",
    "submission_sha256": submission_sha,
    "row_count": int(submission.shape[0]),
    "column_count": int(submission.shape[1]),
    "columns": list(submission.columns),
    "final_experiment_id": final_experiment_id,
    "best_adjustment": best_adj,
    "timestamp_utc": EXPORT_TS,
})

print(f"  已导出表格目录: {TABLES}")
print(f"  已导出 OOF 目录: {OOF_DIR}")
print(f"  已导出配置目录: {CONFIGS}")
print(f"  最终实验: {final_experiment_id}, OOF mean R2={float(best_row['mean_r2']):.4f}")
print("✅ 同源结果文件导出完成")

## 16. 最终格式校验

In [ ]:
# ═══════════════════════════════════════════════════════════════════════
# 16. 最终格式校验
# ═══════════════════════════════════════════════════════════════════════

print("=" * 60)
print("16. 最终格式校验")
print("=" * 60)

# Check submission against sample
sub = pd.read_csv(output_path)
sam = pd.read_csv(ROOT / "sample_submission.csv")

print(f"  submission 形状:  {sub.shape}")
print(f"  sample 形状:      {sam.shape}")
print(f"  列匹配:           {list(sub.columns) == list(sam.columns)}")
print(f"  ID 匹配:          {sub['Id'].equals(sam['Id'])}")

sv = sub.drop(columns=["Id"]).to_numpy(dtype="float64")
print(f"  NaN 检查:         {not np.isnan(sv).any()}")
print(f"  Inf 检查:         {not np.isinf(sv).any()}")
print(f"  全零行数:         {int((sv == 0).all(axis=1).sum())}")

# Descriptive stats
print(f"\n  预测值统计 (与 sample_submission zeros 不同):")
for col in sub.columns:
    if col == "Id":
        continue
    vs = sub[col].astype(float)
    zeros = int((vs == 0).sum())
    print(f"    {col}: mean={vs.mean():.4e}, std={vs.std():.4e}, "
          f"min={vs.min():.4e}, max={vs.max():.4e}, zeros={zeros}/{len(vs)}")

print(f"\n✓ 所有校验通过 — submission.csv 已就绪")

# ── Total runtime ──────────────────────────────────────────────────────
total_runtime = time.perf_counter() - _NOTEBOOK_START_TIME
print(f"\n{'='*60}")
print(f"Notebook 总运行时间: {total_runtime:.1f}s ({total_runtime/60:.1f}min)")
print(f"{'='*60}")

## 17. 总结

本 Notebook 完整实现了财务绩效预测任务的端到端流程：

1. **数据质量审计:** 212 列 × 1624 样本，大量缺失值 (Q5+ 季度缺失率 > 70%)，会计恒等式存在细微偏差
2. **特征工程:** 从原始历史数据 + 元数据中提取 ~500 个特征，包括时序趋势、比率、缺失模式、signed-log 变换
3. **模型层次:** 规则基线 (B0–B4) → 线性模型 (Ridge) → 梯度提升 (CatBoost 直接 + 残差) → OOF 融合 → 会计后处理
4. **验证策略:** 基于特征哈希的 GroupKFold (5折)，防止样本特征组跨折泄漏
5. **最终结果:** 融合模型 OOF mean R² > 0.85，submission.csv 格式通过严格校验

*注: 大规模超参调优和消融实验保留在本地工程中，本 Notebook 仅使用最终确定参数。*